In [1]:
import os
import re
import glob
import json
import os
import re
import os
import glob
import numpy as np
import pandas as pd
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from types import SimpleNamespace

import good

In [2]:
state = "CA"
def load_total_costs_from_results(results_dir, scenarios=None, scenario_label=None):
    records = []

    total_files = glob.glob(os.path.join(results_dir, "**", "*_total_cost.csv"), recursive=True)
    summary_files = glob.glob(os.path.join(results_dir, "**", "*_summary.csv"), recursive=True)
    files = sorted(total_files + summary_files)

    print(f"Found {len(files)} candidate files in {results_dir}")

    seen = set()

    for fp in files:
        try:
            temp = pd.read_csv(fp)
            if temp.empty:
                continue

            row = temp.iloc[0].to_dict()

            fname = os.path.basename(fp)
            folder = os.path.basename(os.path.dirname(fp))

            scenario_id = int(fname.split("_")[0][1:])

            # prefer total_cost over summary
            if scenario_id in seen and fp.endswith("_summary.csv"):
                continue

            cfg = {}
            cfg["scenario_id"] = scenario_id
            cfg["scenario_tag"] = row.get("scenario_tag", "")
            cfg["objective_value"] = float(row["objective_value"])
            cfg["file_path"] = fp

            base_dir_name = os.path.basename(os.path.normpath(results_dir)).lower()

            if "_slow_" in f"_{base_dir_name}_":
                cfg["adoption"] = "Slow"
            elif "_fast_" in f"_{base_dir_name}_":
                cfg["adoption"] = "Fast"
            elif "_mid_" in f"_{base_dir_name}_":
                cfg["adoption"] = "Mid"
            else:
                cfg["adoption"] = "Unknown"

            # NEW: Add scenario label from directory
            if scenario_label is None:
                # Extract label from directory name (e.g., "flex", "midnight")
                parts = base_dir_name.split("_")
                cfg["scenario_label"] = parts[-1] if len(parts) > 0 else "default"
            else:
                cfg["scenario_label"] = scenario_label

            # Extract v1g_share and v2g_share from folder name
            m_v1g = re.search(r"v1g(\d+)", folder, re.IGNORECASE)
            m_v2g = re.search(r"v2g(\d+)", folder, re.IGNORECASE)

            v1g_val = int(m_v1g.group(1)) if m_v1g else 0
            v2g_val = int(m_v2g.group(1)) if m_v2g else 0

            cfg["v1g_share"] = v1g_val / 100.0
            cfg["v2g_share"] = v2g_val / 100.0

            # Determine participation level (10% or 30%)
            total_participation = v1g_val + v2g_val
            if total_participation == 0:
                cfg["participation"] = "0%"
                cfg["group"] = "Base only"
            elif total_participation == 10:
                cfg["participation"] = "10%"
            elif total_participation == 30:
                cfg["participation"] = "30%"
            elif total_participation == 50:
                cfg["participation"] = "50%"
            else:
                cfg["participation"] = f"{total_participation}%"

            # Determine group
            if v1g_val > 0 and v2g_val > 0:
                cfg["group"] = "V1G+V2G"
            elif v1g_val > 0:
                cfg["group"] = "V1G"
            elif v2g_val > 0:
                cfg["group"] = "V2G"
            else:
                cfg["group"] = "Base only"

            m = re.search(r"rps(\d+)", folder)
            cfg["rps_ratio"] = int(m.group(1)) / 100 if m else None

            m = re.search(r"bcapex(\d+)", folder)
            if m:
                batt = int(m.group(1))
                cfg["batt_capex_label"] = f"${batt}/kWh"
            else:
                cfg["batt_capex_label"] = None

            records = [r for r in records if r["scenario_id"] != scenario_id]
            records.append(cfg)
            seen.add(scenario_id)

        except Exception as e:
            print(f"Could not read {fp}: {e}")

    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError(f"No valid total cost or summary files found in: {results_dir}")

    df = df.sort_values("scenario_id").reset_index(drop=True)
    print("Available scenario IDs:", sorted(df["scenario_id"].unique().tolist()))
    print("Participation levels found:", df["participation"].dropna().unique())

    return df

def load_cost_components_from_results(results_dir, scenario_label=None):
    records = []

    # Use total_cost/summary reader as metadata source
    meta_df = load_total_costs_from_results(results_dir, scenarios=None, scenario_label=scenario_label)
    meta_map = {
        int(row["scenario_id"]): row
        for _, row in meta_df.iterrows()
    }

    component_files = glob.glob(
        os.path.join(results_dir, "**", "*_cost_components.csv"),
        recursive=True
    )

    print(f"Found {len(component_files)} cost component files in {results_dir}")

    for fp in sorted(component_files):
        try:
            temp = pd.read_csv(fp)
            if temp.empty:
                continue

            row = temp.iloc[0].to_dict()
            fname = os.path.basename(fp)

            scenario_id = int(fname.split("_")[0][1:])

            if scenario_id not in meta_map:
                print(f"Could not find metadata match for cost component file: {fp}")
                continue

            meta = meta_map[scenario_id]

            cfg = {}
            cfg["scenario_id"] = scenario_id
            cfg["scenario_tag"] = meta.get("scenario_tag", "")
            cfg["objective_value"] = float(row.get("objective_value", meta.get("objective_value", 0.0)))

            cfg["capex_total"] = float(row.get("capex_total", 0.0))
            cfg["asset_opex_total"] = float(row.get("asset_opex_total", 0.0))
            cfg["line_opex_total"] = float(row.get("line_opex_total", 0.0))
            cfg["fixed_total"] = float(row.get("fixed_total", 0.0))
            cfg["penalty_total"] = float(row.get("penalty_total", 0.0))
            cfg["other_gap"] = float(row.get("other_gap", 0.0))
            cfg["file_path"] = fp

            # Bring all plotting metadata from total_cost/summary loader
            cfg["adoption"] = meta.get("adoption", "Unknown")
            cfg["scenario_label"] = meta.get("scenario_label", scenario_label if scenario_label is not None else "default")
            cfg["v1g_share"] = float(meta.get("v1g_share", 0.0))
            cfg["v2g_share"] = float(meta.get("v2g_share", 0.0))
            cfg["participation"] = meta.get("participation", "0%")
            cfg["group"] = meta.get("group", "Base only")
            cfg["rps_ratio"] = meta.get("rps_ratio", None)

            batt_label = meta.get("batt_capex_label", None)
            if batt_label is not None:
                m = re.search(r"(\d+)", str(batt_label))
                cfg["batt_capex_num"] = float(m.group(1)) if m else None
            else:
                cfg["batt_capex_num"] = None

            records.append(cfg)

        except Exception as e:
            print(f"Could not read {fp}: {e}")

    df = pd.DataFrame(records)

    if not df.empty:
        print("Cost component scenario IDs:", sorted(df["scenario_id"].unique().tolist()))
        if "batt_capex_num" in df.columns:
            print("Cost component battery capex values:", sorted(df["batt_capex_num"].dropna().unique().tolist()))

    return df

component_colors = {
    "capex": "#7B6D8D",
    "asset_opex": "#59A14F",
    "line_opex": "#E15759",
    "other": "#C7C7C7",
}

def plot_total_cost_bar_by_rps_with_participation(
    results_dirs_dict,
    batt_capex=150,
    scenarios=None,
    save_path=None,
    xlim_by_rps=None,
    rps_order_list=None,
    baseline_order=None,
    baseline_scenario_labels=None,
    program_scenario_label=None,
    selected_groups=None,
    selected_participation=None,
):
    """
    results_dirs_dict: dict mapping scenario_label -> list of directories for that scenario
                       e.g., {"flex": [dir1, dir2, dir3], "midnight": [dir1, dir2, dir3]}

    baseline_order: list of scenario labels in desired order for baseline scenarios
                    e.g., ["midnight", "flex", "arrive"]

    baseline_scenario_labels: list of scenario labels to show for Base only rows
                              e.g., ["midnight", "flex", "arrive"]

    program_scenario_label: one scenario label to use for V1G / V2G / V1G+V2G rows
                            e.g., "arrive"

    selected_groups: optional list of groups to include
                     e.g., ["Base only", "V1G", "V2G", "V1G+V2G"]

    selected_participation: optional list of participation labels
                            e.g., ["10%", "30%", "50%"]
    """

    from matplotlib.patches import Patch

    if rps_order_list is None:
        rps_order_list = [50, 60, 70, 80]

    # -----------------------------
    # Figure out which scenario labels are needed
    # -----------------------------
    requested_labels = set(results_dirs_dict.keys())

    if baseline_scenario_labels is not None:
        requested_labels = requested_labels.intersection(set(baseline_scenario_labels))

    if program_scenario_label is not None:
        requested_labels = requested_labels.union({program_scenario_label})

    results_dirs_dict = {
        k: v for k, v in results_dirs_dict.items()
        if k in requested_labels
    }

    if len(results_dirs_dict) == 0:
        raise ValueError("No scenario labels left after filtering.")

    # -----------------------------
    # Load cost-component data
    # -----------------------------
    df_comp_list = []
    for scenario_label, dirs in results_dirs_dict.items():
        for rd in dirs:
            df_comp_temp = load_cost_components_from_results(rd, scenario_label=scenario_label)
            if not df_comp_temp.empty:
                df_comp_list.append(df_comp_temp)

    if len(df_comp_list) == 0:
        raise ValueError("No *_cost_components.csv files found.")

    df_comp = pd.concat(df_comp_list, ignore_index=True)
    df_comp = df_comp[df_comp["batt_capex_num"] == batt_capex].copy()

    if selected_participation is not None:
        df_comp = df_comp[df_comp["participation"].isin(selected_participation)].copy()

    if selected_groups is not None:
        df_comp = df_comp[df_comp["group"].isin(selected_groups)].copy()

    if df_comp.empty:
        raise ValueError(f"No cost component rows found for battery capex = {batt_capex}")

    df_comp["rps_percent"] = (df_comp["rps_ratio"] * 100).round().astype(int)

    for col in [
        "capex_total",
        "asset_opex_total",
        "line_opex_total",
        "fixed_total",
        "penalty_total",
        "other_gap",
        "objective_value",
    ]:
        df_comp[col + "_mil"] = df_comp[col] / 1e6

    df_comp["other_total_mil"] = (
        df_comp["fixed_total_mil"]
        + df_comp["penalty_total_mil"]
        + df_comp["other_gap_mil"]
    )

    # -----------------------------
    # Load total cost data
    # -----------------------------
    df_list = []
    for scenario_label, dirs in results_dirs_dict.items():
        for rd in dirs:
            df_temp = load_total_costs_from_results(rd, scenarios, scenario_label=scenario_label)
            df_list.append(df_temp)

    if len(df_list) == 0:
        raise ValueError("No total cost / summary data found.")

    df = pd.concat(df_list, ignore_index=True)
    df["batt_capex_num"] = df["batt_capex_label"].str.extract(r"(\d+)").astype(float)
    df["rps_percent"] = (df["rps_ratio"] * 100).round().astype(int)
    df = df[df["batt_capex_num"] == batt_capex].copy()

    if selected_participation is not None:
        df = df[df["participation"].isin(selected_participation) | (df["group"] == "Base only")].copy()

    if selected_groups is not None:
        df = df[df["group"].isin(selected_groups)].copy()

    if df.empty:
        raise ValueError(f"No rows found for battery capex = {batt_capex}")

    # -----------------------------
    # Split baseline vs program rows
    # -----------------------------
    baseline_df = df[df["group"] == "Base only"].copy()
    program_df = df[df["group"].isin(["V1G", "V2G", "V1G+V2G"])].copy()

    baseline_comp_df = df_comp[df_comp["group"] == "Base only"].copy()
    program_comp_df = df_comp[df_comp["group"].isin(["V1G", "V2G", "V1G+V2G"])].copy()

    if baseline_scenario_labels is not None:
        baseline_df = baseline_df[baseline_df["scenario_label"].isin(baseline_scenario_labels)].copy()
        baseline_comp_df = baseline_comp_df[baseline_comp_df["scenario_label"].isin(baseline_scenario_labels)].copy()

    if program_scenario_label is not None:
        program_df = program_df[program_df["scenario_label"] == program_scenario_label].copy()
        program_comp_df = program_comp_df[program_comp_df["scenario_label"] == program_scenario_label].copy()

    df_plot = pd.concat([baseline_df, program_df], ignore_index=True)
    df_comp_plot = pd.concat([baseline_comp_df, program_comp_df], ignore_index=True)

    if df_plot.empty:
        raise ValueError("No rows left after applying baseline/program scenario filters.")

    df_plot["objective_value_mil"] = df_plot["objective_value"] / 1e6

    rps_order = rps_order_list
    group_order_default = ["Base only", "V1G", "V1G+V2G", "V2G"]

    if selected_groups is not None:
        group_order = [g for g in group_order_default if g in selected_groups]
    else:
        group_order = group_order_default

    # baseline scenario order
    all_baseline_scenarios = df_plot[df_plot["group"] == "Base only"]["scenario_label"].unique()

    if baseline_order is not None:
        baseline_scenarios = [s for s in baseline_order if s in all_baseline_scenarios]
        remaining = [s for s in all_baseline_scenarios if s not in baseline_order]
        baseline_scenarios.extend(sorted(remaining))
    else:
        baseline_scenarios = sorted(all_baseline_scenarios)

    print(f"Baseline scenarios shown: {baseline_scenarios}")
    print(f"Program scenario shown: {program_scenario_label}")

    # -----------------------------
    # Figure layout
    # -----------------------------
    fig = plt.figure(figsize=(16, 12), facecolor="white")
    gs = fig.add_gridspec(2, 4, hspace=0.3, wspace=0.2)

    ax1 = fig.add_subplot(gs[0, 0:2])
    ax2 = fig.add_subplot(gs[0, 2:4])
    ax3 = fig.add_subplot(gs[1, 1:3])

    axes = [ax1, ax2, ax3]

    for ax in axes:
        ax.set_facecolor("white")
        ax.tick_params(axis="both", colors="black", labelcolor="black")
        ax.xaxis.label.set_color("black")
        ax.yaxis.label.set_color("black")
        ax.title.set_color("black")

    bar_width = 0.45
    bar_spacing = 0.05
    group_spacing = 0.6

    participation_order_default = ["10%", "30%", "50%"]
    if selected_participation is not None:
        participation_order = [p for p in participation_order_default if p in selected_participation]
    else:
        participation_order = participation_order_default

    # -----------------------------
    # Helper to get Mid component split
    # -----------------------------
    def get_mid_components(mid_row, mid_val):
        mid_comp = df_comp_plot[
            (df_comp_plot["scenario_id"] == mid_row["scenario_id"]) &
            (df_comp_plot["scenario_label"] == mid_row["scenario_label"]) &
            (df_comp_plot["adoption"] == "Mid")
        ]

        if not mid_comp.empty:
            capex_m = mid_comp["capex_total_mil"].iloc[0]
            asset_opex_m = mid_comp["asset_opex_total_mil"].iloc[0]
            line_opex_m = mid_comp["line_opex_total_mil"].iloc[0]
            other_m = mid_comp["other_total_mil"].iloc[0]
        else:
            capex_m = 0.0
            asset_opex_m = 0.0
            line_opex_m = 0.0
            other_m = mid_val

        return capex_m, asset_opex_m, line_opex_m, other_m

    # -----------------------------
    # Helper to draw stacked Mid bar
    # -----------------------------
    def draw_stacked_mid_bar(ax, current_y, capex_m, asset_opex_m, line_opex_m, other_m):
        left0 = 0.0

        ax.barh(
            current_y, capex_m, left=left0, height=bar_width,
            color=component_colors["capex"], edgecolor="black", linewidth=0.6, zorder=2
        )
        left0 += capex_m

        ax.barh(
            current_y, asset_opex_m, left=left0, height=bar_width,
            color=component_colors["asset_opex"], edgecolor="black", linewidth=0.6, zorder=2
        )
        left0 += asset_opex_m

        ax.barh(
            current_y, line_opex_m, left=left0, height=bar_width,
            color=component_colors["line_opex"], edgecolor="black", linewidth=0.6, zorder=2
        )
        left0 += line_opex_m

        ax.barh(
            current_y, other_m, left=left0, height=bar_width,
            color=component_colors["other"], edgecolor="black", linewidth=0.6,
            zorder=2, hatch="///"
        )

    # -----------------------------
    # Helper to label stacked segments
    # -----------------------------
    def label_segments(ax, current_y, capex_m, asset_opex_m, line_opex_m, other_m):
        segments = [
            ("capex", capex_m),
            ("opex", asset_opex_m),
            ("line", line_opex_m),
            ("other", other_m),
        ]
    # ----------------------------
    # Plot each RPS panel
    # -----------------------------
    for ax_idx, rps in enumerate(rps_order):
        if ax_idx >= len(axes):
            break

        ax = axes[ax_idx]
        sub = df_plot[df_plot["rps_percent"] == rps].copy()

        if sub.empty:
            ax.set_title(f"RPS {rps}%", color="black", fontsize=18)
            ax.grid(True, axis="x", linestyle="--", alpha=0.3)
            continue

        if xlim_by_rps is not None and rps in xlim_by_rps:
            x_min, x_max = xlim_by_rps[rps]
        else:
            vals = sub["objective_value_mil"].dropna().values
            if len(vals) == 0:
                x_min, x_max = 0, 1
            else:
                x_min = 0
                x_max = vals.max()
                pad = 0.12 * (x_max - x_min) if x_max > x_min else 0.5
                x_max += pad

        x_range = x_max - x_min

        y_positions = []
        y_labels = []
        current_y = 0

        for grp in group_order:
            grp_data = sub[sub["group"] == grp]

            if grp == "Base only":
                for scenario_label in baseline_scenarios:
                    scenario_data = grp_data[grp_data["scenario_label"] == scenario_label]
                    mid_data = scenario_data[scenario_data["adoption"] == "Mid"]

                    if mid_data.empty:
                        continue

                    mid_row = mid_data.iloc[0]
                    mid_val = mid_row["objective_value_mil"]

                    slow_data = scenario_data[scenario_data["adoption"] == "Slow"]
                    fast_data = scenario_data[scenario_data["adoption"] == "Fast"]

                    slow_val = slow_data["objective_value_mil"].iloc[0] if not slow_data.empty else mid_val
                    fast_val = fast_data["objective_value_mil"].iloc[0] if not fast_data.empty else mid_val

                    capex_m, asset_opex_m, line_opex_m, other_m = get_mid_components(mid_row, mid_val)

                    draw_stacked_mid_bar(ax, current_y, capex_m, asset_opex_m, line_opex_m, other_m)
                    label_segments(ax, current_y, capex_m, asset_opex_m, line_opex_m, other_m)

                    is_non_monotonic = not (slow_val <= mid_val <= fast_val)

                    if is_non_monotonic:
                        low_val = min(slow_val, mid_val, fast_val)
                        high_val = max(slow_val, mid_val, fast_val)
                        left_err = mid_val - low_val
                        right_err = high_val - mid_val
                        whisker_color = "#D62728"
                    else:
                        left_err = mid_val - slow_val
                        right_err = fast_val - mid_val
                        whisker_color = "black"

                    ax.errorbar(
                        x=mid_val,
                        y=current_y,
                        xerr=np.array([[left_err], [right_err]]),
                        fmt="none",
                        ecolor=whisker_color,
                        elinewidth=2.2 if is_non_monotonic else 1.5,
                        capsize=4,
                        capthick=2.2 if is_non_monotonic else 1.5,
                        zorder=3
                    )

                    ax.text(
                        max(0.01 * mid_val, x_min + 0.01 * x_range),
                        current_y,
                        scenario_label.upper(),
                        va="center",
                        ha="left",
                        fontsize=15,
                        fontweight="bold",
                        color="white",
                        zorder=4
                    )

                    x_offset = 0.01 * x_range
                    ax.text(
                        slow_val - x_offset,
                        current_y,
                        f"${slow_val:.2f}M",
                        ha="right",
                        va="center",
                        fontsize=15,
                        color="black",
                        clip_on=True
                    )
                    ax.text(
                        fast_val + x_offset,
                        current_y,
                        f"${fast_val:.2f}M",
                        ha="left",
                        va="center",
                        fontsize=15,
                        color="black",
                        clip_on=True
                    )

                    y_positions.append(current_y)
                    y_labels.append("Base" if scenario_label == baseline_scenarios[0] else "")
                    current_y += bar_width + bar_spacing

                current_y += group_spacing - (bar_width + bar_spacing)

            else:
                first_row_for_group = True

                for part_level in participation_order:
                    part_data = grp_data[grp_data["participation"] == part_level]
                    mid_data = part_data[part_data["adoption"] == "Mid"]

                    if mid_data.empty:
                        continue

                    mid_row = mid_data.iloc[0]
                    mid_val = mid_row["objective_value_mil"]

                    slow_data = part_data[part_data["adoption"] == "Slow"]
                    fast_data = part_data[part_data["adoption"] == "Fast"]

                    slow_val = slow_data["objective_value_mil"].iloc[0] if not slow_data.empty else mid_val
                    fast_val = fast_data["objective_value_mil"].iloc[0] if not fast_data.empty else mid_val

                    capex_m, asset_opex_m, line_opex_m, other_m = get_mid_components(mid_row, mid_val)

                    draw_stacked_mid_bar(ax, current_y, capex_m, asset_opex_m, line_opex_m, other_m)
                    label_segments(ax, current_y, capex_m, asset_opex_m, line_opex_m, other_m)

                    is_non_monotonic = not (slow_val <= mid_val <= fast_val)

                    if is_non_monotonic:
                        low_val = min(slow_val, mid_val, fast_val)
                        high_val = max(slow_val, mid_val, fast_val)
                        left_err = mid_val - low_val
                        right_err = high_val - mid_val
                        whisker_color = "#D62728"
                    else:
                        left_err = mid_val - slow_val
                        right_err = fast_val - mid_val
                        whisker_color = "black"

                    ax.errorbar(
                        x=mid_val,
                        y=current_y,
                        xerr=np.array([[left_err], [right_err]]),
                        fmt="none",
                        ecolor=whisker_color,
                        elinewidth=2.2 if is_non_monotonic else 1.5,
                        capsize=4,
                        capthick=2.2 if is_non_monotonic else 1.5,
                        zorder=3
                    )

                    ax.text(
                        max(0.01 * mid_val, x_min + 0.01 * x_range),
                        current_y,
                        f"{part_level} participation",
                        va="center",
                        ha="left",
                        fontsize=15,
                        fontweight="bold",
                        color="white",
                        zorder=4
                    )

                    x_offset = 0.01 * x_range
                    ax.text(
                        slow_val - x_offset,
                        current_y,
                        f"${slow_val:.2f}M",
                        ha="right",
                        va="center",
                        fontsize=15,
                        color="black",
                        clip_on=True
                    )
                    ax.text(
                        fast_val + x_offset,
                        current_y,
                        f"${fast_val:.2f}M",
                        ha="left",
                        va="center",
                        fontsize=15,
                        color="black",
                        clip_on=True
                    )

                    y_positions.append(current_y)
                    y_labels.append(grp if first_row_for_group else "")
                    first_row_for_group = False

                    current_y += bar_width + bar_spacing

                current_y += group_spacing - (bar_width + bar_spacing)

        ax.set_title(f"RPS {rps}%", color="black", fontsize=18)
        ax.set_xlim(x_min, x_max)
        ax.set_yticks(y_positions)
        ax.set_yticklabels(y_labels, color="black", fontsize=16)
        ax.invert_yaxis()
        ax.set_xlabel("Total system cost (Million $)", color="black", fontsize=16)
        ax.grid(True, axis="x", linestyle="--", alpha=0.3)

        for spine in ax.spines.values():
            spine.set_color("black")

    axes[0].set_ylabel("Charging Scenarios", fontsize=16)
    axes[2].set_ylabel("Charging Scenarios", fontsize=16)

    title_program = program_scenario_label if program_scenario_label is not None else "all"
    fig.suptitle(
        f"Total system cost by RPS and participation level\n"
        f"Baseline scenarios shown separately, program rows from '{title_program}'\n"
        f"Battery capex = ${int(batt_capex)}/kWh",
        fontsize=15,
        y=0.96,
        color="black"
    )

    component_legend = [
        Patch(facecolor=component_colors["capex"], edgecolor="black", label="Asset CAPEX"),
        Patch(facecolor=component_colors["asset_opex"], edgecolor="black", label="Asset OPEX"),
        Patch(facecolor=component_colors["line_opex"], edgecolor="black", label="Line OPEX"),
        Patch(facecolor=component_colors["other"], edgecolor="black", hatch="///", label="Penalty / Fixed / Other"),
    ]

    legend = fig.legend(
        handles=component_legend,
        loc="lower center",
        ncol=4,
        frameon=True,
        bbox_to_anchor=(0.5, 0.01),
        fontsize=15
    )

    legend.get_frame().set_facecolor("white")
    legend.get_frame().set_edgecolor("black")
    legend.get_frame().set_linewidth(0.8)

    for text in legend.get_texts():
        text.set_color("black")
        text.set_fontsize(11)

    plt.tight_layout(rect=[0.03, 0.06, 1, 0.94])

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    return df_plot, fig, axes


# Usage example with baseline order specified
xlim_by_rps = {
    50: (0.0, 200),
    60: (0.0, 200),
    70: (0.0, 220),
}

# Define directories with labels
results_dirs_dict = {
    "flex": [
        f"scenario_results_2030_slow_even_{state}_flex",
        f"scenario_results_2030_mid_even_{state}_flex",
        f"scenario_results_2030_fast_even_{state}_flex",
    ],
    "midnight": [
        f"scenario_results_2030_slow_even_{state}_midnight",
        f"scenario_results_2030_mid_even_{state}_midnight",
        f"scenario_results_2030_fast_even_{state}_midnight",
    ],
    "arrive": [
        f"scenario_results_2030_slow_even_{state}_arrive",
        f"scenario_results_2030_mid_even_{state}_arrive",
        f"scenario_results_2030_fast_even_{state}_arrive",
    ],

}

# NEW: Specify the order - midnight will appear first (top)
baseline_order = ["midnight", "flex"]

df_cost_range, fig, axes = plot_total_cost_bar_by_rps_with_participation(
    results_dirs_dict=results_dirs_dict,
    batt_capex=150,
    save_path="figures/total_cost_arrive_programs.png",
    xlim_by_rps=xlim_by_rps,
    rps_order_list=[50, 60, 70],
    baseline_order=["midnight", "arrive","flex"],
    baseline_scenario_labels=["midnight", "arrive","flex"],
    program_scenario_label="flex",
)

df_cost_range, fig, axes = plot_total_cost_bar_by_rps_with_participation(
    results_dirs_dict=results_dirs_dict,
    batt_capex=250,
    save_path="figures/total_cost_arrive_programs.png",
    xlim_by_rps=xlim_by_rps,
    rps_order_list=[50, 60, 70],
    baseline_order=["midnight", "arrive","flex"],
    baseline_scenario_labels=["midnight", "arrive","flex"],
    program_scenario_label="flex",
)

Found 0 candidate files in scenario_results_2030_slow_even_CA_flex


ValueError: No valid total cost or summary files found in: scenario_results_2030_slow_even_CA_flex

In [ ]:
def load_total_costs_from_results(results_dir, scenarios=None):
    records = []

    total_files = glob.glob(os.path.join(results_dir, "**", "*_total_cost.csv"), recursive=True)
    summary_files = glob.glob(os.path.join(results_dir, "**", "*_summary.csv"), recursive=True)
    files = sorted(total_files + summary_files)

    print(f"Found {len(files)} candidate files")

    seen = set()

    for fp in files:
        try:
            temp = pd.read_csv(fp)
            if temp.empty:
                continue

            row = temp.iloc[0].to_dict()

            fname = os.path.basename(fp)
            folder = os.path.basename(os.path.dirname(fp))

            scenario_id = int(fname.split("_")[0][1:])

            # prefer total_cost over summary
            if scenario_id in seen and fp.endswith("_summary.csv"):
                continue

            cfg = {}
            cfg["scenario_id"] = scenario_id
            cfg["scenario_tag"] = row.get("scenario_tag", "")
            cfg["objective_value"] = float(row["objective_value"])
            cfg["file_path"] = fp

            base_dir_name = os.path.basename(os.path.normpath(results_dir)).lower()

            if "_slow_" in f"_{base_dir_name}_":
                cfg["adoption"] = "Slow"
            elif "_fast_" in f"_{base_dir_name}_":
                cfg["adoption"] = "Fast"
            elif "_mid_" in f"_{base_dir_name}_":
                cfg["adoption"] = "Mid"
            else:
                cfg["adoption"] = "Unknown"

            # NEW: Extract v1g_share and v2g_share from folder name
            m_v1g = re.search(r"v1g(\d+)", folder, re.IGNORECASE)
            m_v2g = re.search(r"v2g(\d+)", folder, re.IGNORECASE)

            v1g_val = int(m_v1g.group(1)) if m_v1g else 0
            v2g_val = int(m_v2g.group(1)) if m_v2g else 0

            cfg["v1g_share"] = v1g_val / 100.0
            cfg["v2g_share"] = v2g_val / 100.0

            # Determine participation level (10% or 30%)
            total_participation = v1g_val + v2g_val
            if total_participation == 0:
                cfg["participation"] = "0%"
                cfg["group"] = "Base only"
            elif total_participation == 10:
                cfg["participation"] = "10%"
            elif total_participation == 30:
                cfg["participation"] = "30%"
            elif total_participation == 50:
                cfg["participation"] = "50%"
            else:
                cfg["participation"] = f"{total_participation}%"

            # Determine group
            if "_Base_only_" in folder:
                cfg["group"] = "Base only"
            elif v1g_val > 0 and v2g_val > 0:
                cfg["group"] = "V1G+V2G"
            elif v1g_val > 0:
                cfg["group"] = "V1G"
            elif v2g_val > 0:
                cfg["group"] = "V2G"
            else:
                cfg["group"] = "Base only"

            # rps
            m = re.search(r"rps(\d+)", folder)
            cfg["rps_ratio"] = int(m.group(1)) / 100 if m else None

            # battery capex label
            m = re.search(r"bcapex(\d+)", folder)
            if m:
                batt = int(m.group(1))
                cfg["batt_capex_label"] = f"${batt}/kWh"
            else:
                cfg["batt_capex_label"] = None

            records = [r for r in records if r["scenario_id"] != scenario_id]
            records.append(cfg)
            seen.add(scenario_id)

        except Exception as e:
            print(f"Could not read {fp}: {e}")

    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError(f"No valid total cost or summary files found in: {results_dir}")

    df = df.sort_values("scenario_id").reset_index(drop=True)
    print("Participation levels found:", df["participation"].unique())

    return df


def plot_total_cost_delta_vs_rps(
    results_dirs,
    scenarios=None,
    save_path=None,
):
    df_list = [load_total_costs_from_results(rd, scenarios) for rd in results_dirs]
    df = pd.concat(df_list, ignore_index=True)

    df["group_plot"] = df["group"].replace({"Base only": "Base"})
    df["batt_capex_num"] = df["batt_capex_label"].str.extract(r"(\d+)").astype(int)
    df["rps_percent"] = df["rps_ratio"] * 100

    # build baseline table from Base scenarios
    base_df = df[df["group_plot"] == "Base"][[
        "adoption", "rps_ratio", "batt_capex_label", "objective_value"
    ]].rename(columns={"objective_value": "base_cost"})

    # merge matching baseline into every row
    df = df.merge(
        base_df,
        on=["adoption", "rps_ratio", "batt_capex_label"],
        how="left"
    )

    if df["base_cost"].isna().any():
        missing = df[df["base_cost"].isna()][["scenario_id", "rps_ratio", "batt_capex_label"]]
        raise ValueError(f"Missing matching Base scenario for:\n{missing}")

    df["delta_cost_percent"] = 100 * (df["objective_value"] - df["base_cost"]) / df["base_cost"]

    df["label_text"] = df.apply(
        lambda r: f"({r['delta_cost_percent']:+.1f}%)",
        axis=1
    )

    # remove Base from plotted lines
    df_plot = df[df["group_plot"] != "Base"].copy()

    # NEW: Create combined group label for legend
    df_plot["group_participation"] = df_plot.apply(
        lambda r: f"{r['group_plot']} {r['participation']}", axis=1
    )

    batt_order = [150, 250]
    group_order = ["V1G", "V2G", "V1G+V2G"]
    participation_order = ["10%", "30%", "50%"]
    adoption_order = ["Slow", "Mid", "Fast"]

    # NEW: Line styles for participation levels
    linestyle_map = {
        "10%": "-",      # solid for 10%
        "30%": "--",     # dashed for 30%
        "50%": "-.",     # dashed for 30%
    }

    # NEW: Colors for adoption × group combinations
    color_map = {
        # Slow (blue family)
        ("Slow", "V1G"): "#9ecae1",       # light blue
        ("Slow", "V1G+V2G"): "#6baed6",   # medium blue
        ("Slow", "V2G"): "#3182bd",       # dark blue

        # Mid (green family)
        ("Mid", "V1G"): "#a1d99b",        # light green
        ("Mid", "V1G+V2G"): "#74c476",    # medium green
        ("Mid", "V2G"): "#31a354",        # dark green

        # Fast (orange family)
        ("Fast", "V1G"): "#fdae6b",       # light orange
        ("Fast", "V1G+V2G"): "#fd8d3c",   # medium orange
        ("Fast", "V2G"): "#e6550d",       # dark orange
    }

    plt.style.use("default")
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True, facecolor="white")
    for ax in axes.flatten():
        ax.set_facecolor("white")
    axes = axes.flatten()

    cluster_offsets = [-50, -35, -20, -5, 10, 25, 40, 55]

    for ax, batt in zip(axes, batt_order):
        subset_batt = df_plot[df_plot["batt_capex_num"] == batt].copy()

        # Plot each combination
        for group in group_order:
            for participation in participation_order:
                for adoption in adoption_order:
                    subset = subset_batt[
                        (subset_batt["group_plot"] == group) &
                        (subset_batt["participation"] == participation) &
                        (subset_batt["adoption"] == adoption)
                    ].copy()

                    if subset.empty:
                        continue

                    subset = subset.sort_values("rps_percent")

                    ax.plot(
                        subset["rps_percent"],
                        subset["delta_cost_percent"],
                        marker="o",
                        linewidth=2,
                        linestyle=linestyle_map[participation],
                        color=color_map[(adoption, group)],
                        label=f"{group} {participation} {adoption}"
                    )

        # Annotate labels
        for x_val, subx in subset_batt.groupby("rps_percent"):
            subx = subx.sort_values("delta_cost_percent").reset_index(drop=True)

            n = len(subx)

            if n <= len(cluster_offsets):
                offsets = cluster_offsets[:n]
            else:
                offsets = list(np.linspace(-60, 60, n))

            for i, (_, row) in enumerate(subx.iterrows()):
                dy = offsets[i]

                dx = 10 if i % 2 == 0 else -10
                ha = "left" if dx > 0 else "right"

                ax.annotate(
                    row["label_text"],
                    xy=(row["rps_percent"], row["delta_cost_percent"]),
                    xytext=(dx, dy),
                    textcoords="offset points",
                    ha=ha,
                    va="center",
                    fontsize=7,
                    color="black",
                    arrowprops=dict(
                        arrowstyle="-",
                        color="black",
                        lw=0.5,
                        shrinkA=0,
                        shrinkB=0,
                        connectionstyle="arc3,rad=0.0"
                    ),
                    bbox=dict(
                        boxstyle="round,pad=0.15",
                        fc="white",
                        ec="gray",
                        alpha=0.9
                    ),
                    zorder=10
                )

        ax.axhline(0, color="black", linewidth=1, linestyle="--")
        ax.set_title(f"Battery capex = ${batt}/kWh")
        ax.set_xlabel("RPS target (%)", color="black")
        ax.grid(True, alpha=0.25, color="gray", linestyle="--", linewidth=0.6)
        ax.tick_params(axis="both", colors="black")
        ax.title.set_color("black")
        ax.xaxis.label.set_color("black")
        ax.yaxis.label.set_color("black")
        for spine in ax.spines.values():
            spine.set_color("black")
        ax.set_ylim(-25.25, 2.25)

    # Create legend
    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))

    fig.legend(
        by_label.values(),
        by_label.keys(),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=6,
        frameon=False,
        fontsize=8
    )

    axes[0].set_ylabel("Change in total cost relative to Base scenario (%)")
    axes[1].set_ylabel("Change in total cost relative to Base scenario (%)")

    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    return df, fig, axes

In [ ]:
with open(f"Examples/scenarios_2030_{state}.json", "r") as f:
    SCENARIOS = json.load(f)
SCENARIOS = {int(k): v for k, v in SCENARIOS.items()}

results_dirs = [
    f"scenario_results_2030_slow_even_{state}_flex",
    f"scenario_results_2030_mid_even_{state}_flex",
    f"scenario_results_2030_fast_even_{state}_flex",
]

df, fig, axes = plot_total_cost_delta_vs_rps(
    results_dirs=results_dirs,
    save_path=f"figures/total_cost_delta_vs_rps_adoption_{state}.png"
)


In [ ]:

graph_json_path_target="Examples/California_new.json"
def parse_scenario_metadata(folder_name):
    meta = {}

    m = re.search(r"^s(\d+)_", folder_name)
    meta["scenario_id"] = int(m.group(1)) if m else None

    # NEW: Extract v1g_share and v2g_share from folder name
    m_v1g = re.search(r"v1g(\d+)", folder_name, re.IGNORECASE)
    m_v2g = re.search(r"v2g(\d+)", folder_name, re.IGNORECASE)

    v1g_val = int(m_v1g.group(1)) if m_v1g else 0
    v2g_val = int(m_v2g.group(1)) if m_v2g else 0

    meta["v1g_share"] = v1g_val / 100.0
    meta["v2g_share"] = v2g_val / 100.0

    # Determine participation level
    total_participation = v1g_val + v2g_val
    if total_participation == 0:
        meta["participation"] = "0%"
        meta["group"] = "Base only"
    elif total_participation == 10:
        meta["participation"] = "10%"
    elif total_participation == 30:
        meta["participation"] = "30%"
    else:
        meta["participation"] = f"{total_participation}%"

    # Determine group
    if "_Base_only_" in folder_name:
        meta["group"] = "Base only"
    elif v1g_val > 0 and v2g_val > 0:
        meta["group"] = "V1G+V2G"
    elif v1g_val > 0:
        meta["group"] = "V1G"
    elif v2g_val > 0:
        meta["group"] = "V2G"
    else:
        meta["group"] = "Base only"

    m = re.search(r"rps(\d+)", folder_name)
    meta["rps"] = int(m.group(1)) if m else None

    m = re.search(r"bcapex(\d+)", folder_name)
    meta["batt_capex"] = int(m.group(1)) if m else None

    return meta

def energy_balance_table_v2(solution, graph, time_step_s=3600.0, power_unit="W"):
    """
    Energy balance table (inputs from solution_graph + graph metadata).

    power_unit:
      - "W"  : power series are in Watts (recommended for your case)
      - "MW" : power series are in MW

    Output:
      category: demand / generation / slack / TOTALS
      fuel
      energy_gwh
      avg_gw
    """
    any_node = next(iter(solution._node.values()))
    T = len(any_node.get("wastage", any_node.get("shortfall", [])))
    if T == 0:
        raise ValueError("T=0. Solve likely infeasible or no series stored.")

    dt_h = time_step_s / 3600.0

    # unit conversions
    if power_unit.upper() == "W":
        to_GW = 1e9
        to_GWh = 1e9
    elif power_unit.upper() == "MW":
        to_GW = 1e3
        to_GWh = 1e3
    else:
        raise ValueError("power_unit must be 'W' or 'MW'")

    rows = []

    # ---- slack at node level
    for region, node in solution._node.items():
        wast = np.asarray(node.get("wastage", [0]*T), dtype=float).ravel()
        shrt = np.asarray(node.get("shortfall", [0]*T), dtype=float).ravel()

        rows.append({"category":"slack","fuel":"wastage","energy_gwh": wast.sum()*dt_h/to_GWh, "avg_gw": wast.mean()/to_GW})
        rows.append({"category":"slack","fuel":"shortfall","energy_gwh": shrt.sum()*dt_h/to_GWh,"avg_gw": shrt.mean()/to_GW})

    # ---- assets
    for region, node in solution._node.items():
        for handle, a_sol in node["assets"].items():

            if handle not in graph._node[region]["assets"]:
                continue

            meta = graph._node[region]["assets"][handle]
            cls = meta.get("_class", "")
            typ = meta.get("type", "")
            fuel = meta.get("fuel", None) or (typ if typ else cls)

            net = np.asarray(a_sol.get("net", [0]*T), dtype=float).ravel()

                        # Demand loads
            if cls == "Load" and typ == "load":
                demand = -net if net.mean() < 0 else net
                rows.append({
                    "category": "demand",
                    "fuel": "load",
                    "energy_gwh": demand.sum()*dt_h/to_GWh,
                    "avg_gw": demand.mean()/to_GW,
                })
                continue

            # Storage: separate discharge and charge
            if cls == "Store":
                prod = np.asarray(a_sol.get("production", [0]*T), dtype=float).ravel()
                cons = np.asarray(a_sol.get("consumption", [0]*T), dtype=float).ravel()

                handle_lower = str(handle).lower()
                type_lower = str(meta.get("type", "")).lower()
                fuel_lower = str(meta.get("fuel", "")).lower()

                is_v2g = (
                    type_lower == "ev_v2g"
                    or fuel_lower == "ev_v2g"
                    or "ev_v2g" in handle_lower
                )

                discharge_name = "v2g_discharge" if is_v2g else "battery_discharge"
                charge_name = "v2g_charge" if is_v2g else "battery_charge"

                if prod.sum() > 0:
                    rows.append({
                        "category": "generation",
                        "fuel": discharge_name,
                        "energy_gwh": prod.sum()*dt_h/to_GWh,
                        "avg_gw": prod.mean()/to_GW,
                    })

                if cons.sum() > 0:
                    rows.append({
                        "category": "demand",
                        "fuel": charge_name,
                        "energy_gwh": cons.sum()*dt_h/to_GWh,
                        "avg_gw": cons.mean()/to_GW,
                    })
                continue

            # Generation only
            if "net" in a_sol and (cls == "Producer" or (cls == "Load" and typ != "load")):
                gen = np.clip(net, 0, None)
                rows.append({
                    "category": "generation",
                    "fuel": str(fuel),
                    "energy_gwh": gen.sum()*dt_h/to_GWh,
                    "avg_gw": gen.mean()/to_GW,
                })

    df = pd.DataFrame(rows)

    summary = (
        df.groupby(["category","fuel"], as_index=False)[["energy_gwh","avg_gw"]]
        .sum()
        .sort_values(["category","energy_gwh"], ascending=[True, False])
    )

    total_demand = summary.loc[summary["category"]=="demand","energy_gwh"].sum()
    total_gen = summary.loc[summary["category"]=="generation","energy_gwh"].sum()
    total_wast = summary.loc[(summary["category"]=="slack") & (summary["fuel"]=="wastage"),"energy_gwh"].sum()
    total_shrt = summary.loc[(summary["category"]=="slack") & (summary["fuel"]=="shortfall"),"energy_gwh"].sum()

    # Correct balance check now that demand is positive
    balance = total_gen - total_demand - total_wast + total_shrt  # (shortfall behaves like supply of unmet demand)

    totals = pd.DataFrame([
        {"category":"TOTALS","fuel":"demand","energy_gwh": total_demand,"avg_gw": np.nan},
        {"category":"TOTALS","fuel":"generation","energy_gwh": total_gen,"avg_gw": np.nan},
        {"category":"TOTALS","fuel":"wastage","energy_gwh": total_wast,"avg_gw": np.nan},
        {"category":"TOTALS","fuel":"shortfall","energy_gwh": total_shrt,"avg_gw": np.nan},
        {"category":"TOTALS","fuel":"balance_gen_minus_demand","energy_gwh": balance,"avg_gw": np.nan},
    ])

    return pd.concat([summary, totals], ignore_index=True)


def load_solution_as_object(solution_json_path):
    """
    Load saved node-link json and convert to a simple object with ._node
    so your energy_balance_table_v2 can use it.
    """
    with open(solution_json_path, "r") as f:
        data = json.load(f)

    G = nx.node_link_graph(data)
    solution = SimpleNamespace()
    solution._node = {n: G.nodes[n] for n in G.nodes}
    solution.graph = G.graph
    return solution


def estimate_time_step_s_from_solution(solution, default_time_step_s=3600.0):
    """
    Use the actual timestep from the solution if it exists.
    If not available, assume hourly data.
    """

    # Case 1: timestep saved at graph level
    if hasattr(solution, "graph"):
        for key in ["time_step", "time_step_s", "timestep", "dt"]:
            if key in solution.graph:
                return float(solution.graph[key])

    # Case 2: timestep saved inside any node
    any_node = next(iter(solution._node.values()))
    for key in ["time_step", "time_step_s", "timestep", "dt"]:
        if key in any_node:
            return float(any_node[key])

    # Case 3: most GOOD outputs are hourly unless explicitly aggregated
    print(f"No timestep found in solution. Using default_time_step_s = {default_time_step_s}")
    return float(default_time_step_s)


def map_fuel_for_plot_extended(fuel):
    f = str(fuel).strip().lower()

    if f == "battery_discharge":
        return "Battery Discharge"
    elif f == "v2g_discharge":
        return "V2G Discharge"
    elif f == "battery_charge":
        return "Battery Charge"
    elif f == "v2g_charge":
        return "V2G Charge"
    elif f == "solar":
        return "Solar"
    elif f == "wind":
        return "Wind"
    elif f == "nuclear":
        return "Nuclear"
    elif f == "coal":
        return "Coal"
    elif f == "natural gas combined cycle":
        return "CCNG"
    elif f == "natural gas turbine":
        return "Gas Turbine"
    elif f == "hydro":
        return "Hydro"
    elif f == "geothermal":
        return "Geothermal"
    elif f == "biomass":
        return "Biomass"
    elif f == "waste":
        return "Waste"
    elif f == "import":
        return "Import"
    elif f == "oil":
        return "Oil"
    elif f == "non-fossil":
        return "Non-fossil"
    else:
        return "Other"

def collect_curtailment_deltas(
    results_dirs,
    graph_json_path,
    batt_capex_target=150,
    days=30
):
    dfc = collect_estimated_curtailment_from_wastage(
        results_dirs=results_dirs,
        graph_json_path=graph_json_path,
        batt_capex_target=batt_capex_target,
        days=days
    ).copy()

    dfc["resource"] = "Curtailment"
    dfc["energy_gwh"] = dfc["estimated_solar_wind_curtailment_gwh"]

    base = (
        dfc[dfc["group"] == "Base only"]
        .rename(columns={"energy_gwh": "base_energy_gwh"})
        [["adoption", "rps", "batt_capex", "resource", "base_energy_gwh"]]
    )

    comp = dfc[dfc["group"].isin(["V1G", "V2G", "V1G+V2G"])].copy()

    merged = comp.merge(
        base,
        on=["adoption", "rps", "batt_capex", "resource"],
        how="left"
    )

    merged["base_energy_gwh"] = merged["base_energy_gwh"].fillna(0.0)
    merged["delta_gwh"] = merged["energy_gwh"] - merged["base_energy_gwh"]

    return merged[["adoption", "group", "rps", "batt_capex", "resource", "energy_gwh", "base_energy_gwh", "delta_gwh"]]

def generation_summary_for_one_scenario(
    solution_json_path,
    graph,
    default_time_step_s=3600.0
):
    solution = load_solution_as_object(solution_json_path)
    time_step_s = estimate_time_step_s_from_solution(
        solution,
        default_time_step_s=default_time_step_s
    )

    tbl = energy_balance_table_v2(
        solution=solution,
        graph=graph,
        time_step_s=time_step_s,
        power_unit="W"
    )

    gen = tbl[tbl["category"] == "generation"].copy()
    gen["resource"] = gen["fuel"].apply(map_fuel_for_plot_extended)

    # keep battery discharge too
    keep = [
        "Solar", "Wind", "Nuclear", "Coal", "CCNG", "Gas Turbine",
        "Hydro", "Geothermal", "Biomass", "Waste", "Import", "Oil",
        "Non-fossil", "Other", "Battery Discharge"
    ]
    gen = gen[gen["resource"].isin(keep)].copy()

    gen = gen.groupby("resource", as_index=False)["energy_gwh"].sum()
    return gen


def collect_generation_deltas(
    results_dir,
    graph_json_path,
    batt_capex_target=300,
    days=30
):
    graph = good.graph.graph_from_json(graph_json_path)

    folders = sorted(glob.glob(os.path.join(results_dir, "s*")))
    all_rows = []

    for folder in folders:
        folder_name = os.path.basename(folder)
        meta = parse_scenario_metadata(folder_name)

        if meta["batt_capex"] != batt_capex_target:
            continue

        solution_csvs = glob.glob(os.path.join(folder, "*_solution.csv"))
        if not solution_csvs:
            continue

        solution_json_path = solution_csvs[0]

        try:
            gen = generation_summary_from_solution_csv(
                solution_csv_path=solution_csvs[0],
                graph=graph,
                time_step_h=1.0,
            )

            gen["scenario_id"] = meta["scenario_id"]
            gen["group"] = meta["group"]
            gen["participation"] = meta["participation"]  # NEW
            gen["rps"] = meta["rps"]
            gen["batt_capex"] = meta["batt_capex"]

            all_rows.append(gen)

        except Exception as e:
            print(f"Could not process {solution_json_path}: {e}")

    if not all_rows:
        raise ValueError("No valid generation data found.")

    df = pd.concat(all_rows, ignore_index=True)

    base = df[df["group"] == "Base only"].copy()
    base = base.rename(columns={"energy_gwh": "base_energy_gwh"})
    base = base[["rps", "batt_capex", "resource", "base_energy_gwh"]]

    comp = df[df["group"].isin(["V1G", "V2G", "V1G+V2G"])].copy()

    merged = comp.merge(
        base,
        on=["rps", "batt_capex", "resource"],
        how="left"
    )

    merged["base_energy_gwh"] = merged["base_energy_gwh"].fillna(0.0)
    merged["delta_gwh"] = merged["energy_gwh"] - merged["base_energy_gwh"]

    return merged

def collect_generation_deltas_with_scenario_labels(
    state,
    adoption,
    graph_json_path,
    batt_capex_target=150,
    baseline_scenario_label="flex",
    program_scenario_label="flex",
):
    graph = good.graph.graph_from_json(graph_json_path)

    baseline_dir = f"scenario_results_2030_{adoption}_even_{state}_{baseline_scenario_label}"
    program_dir = f"scenario_results_2030_{adoption}_even_{state}_{program_scenario_label}"

    all_rows = []

    # -----------------------------
    # Load baseline rows
    # -----------------------------
    for folder in sorted(glob.glob(os.path.join(baseline_dir, "s*"))):
        folder_name = os.path.basename(folder)
        meta = parse_scenario_metadata(folder_name)

        if meta["batt_capex"] != batt_capex_target:
            continue

        if meta["group"] != "Base only":
            continue

        solution_jsons = glob.glob(os.path.join(folder, "*_solution.json"))
        if not solution_jsons:
            continue

        try:
            gen = generation_summary_from_solution_json(
                solution_json_path=solution_jsons[0],
                graph=graph,
                time_step_h=1.0,
            )

            gen["scenario_id"] = meta["scenario_id"]
            gen["group"] = meta["group"]
            gen["participation"] = meta["participation"]
            gen["rps"] = meta["rps"]
            gen["batt_capex"] = meta["batt_capex"]
            gen["scenario_label"] = baseline_scenario_label

            all_rows.append(gen)

        except Exception as e:
            print(f"Could not process baseline {solution_jsons[0]}: {e}")

    # -----------------------------
    # Load program rows
    # -----------------------------
    for folder in sorted(glob.glob(os.path.join(program_dir, "s*"))):
        folder_name = os.path.basename(folder)
        meta = parse_scenario_metadata(folder_name)

        if meta["batt_capex"] != batt_capex_target:
            continue

        if meta["group"] not in ["V1G", "V2G", "V1G+V2G"]:
            continue

        solution_jsons = glob.glob(os.path.join(folder, "*_solution.json"))

        if not solution_jsons:
            continue

        try:
            gen = generation_summary_from_solution_json(
                solution_json_path=solution_jsons[0],
                graph=graph,
                time_step_h=1.0,
            )

            gen["scenario_id"] = meta["scenario_id"]
            gen["group"] = meta["group"]
            gen["participation"] = meta["participation"]
            gen["rps"] = meta["rps"]
            gen["batt_capex"] = meta["batt_capex"]
            gen["scenario_label"] = program_scenario_label

            all_rows.append(gen)

        except Exception as e:
            print(f"Could not process program {solution_jsons[0]}: {e}")

    if not all_rows:
        raise ValueError("No valid generation data found.")

    df = pd.concat(all_rows, ignore_index=True)

    base = df[df["group"] == "Base only"].copy()
    base = base.rename(columns={"energy_gwh": "base_energy_gwh"})
    base = base[["rps", "batt_capex", "resource", "base_energy_gwh"]]

    comp = df[df["group"].isin(["V1G", "V2G", "V1G+V2G"])].copy()

    merged = comp.merge(
        base,
        on=["rps", "batt_capex", "resource"],
        how="left",
    )

    merged["base_energy_gwh"] = merged["base_energy_gwh"].fillna(0.0)
    merged["delta_gwh"] = merged["energy_gwh"] - merged["base_energy_gwh"]

    return merged

def plot_generation_delta_one_plot_two_sides(
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=7,
    rps_values=None,
    save_path=None,
    state="CA",
    adoption="mid",
    baseline_scenario_label="flex",
    program_scenario_label="flex",
    label_threshold=20,  # Only label bars with abs(val) >= this
):
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    if rps_values is None:
        raise ValueError("Please provide RPS values, for example rps_values=[50, 60, 70]")

    rps_values = list(rps_values)
    n_rps = len(rps_values)

    df = collect_generation_deltas_with_scenario_labels(
    state=state,
    adoption=adoption,
    graph_json_path=graph_json_path,
    batt_capex_target=batt_capex_target,
    baseline_scenario_label=baseline_scenario_label,
    program_scenario_label=program_scenario_label,
    )

    resource_order = [
        "Solar", "Wind", "Nuclear", "Coal", "CCNG", "Gas Turbine",
        "Hydro", "Geothermal", "Biomass", "Waste", "Import", "Oil",
        "Non-fossil", "Other",
    ]

    colors = {
        "Solar": "#f4b400",
        "Wind": "#4a90e2",
        "Nuclear": "#ff1f1f",
        "Coal": "#000000",
        "CCNG": "#8c8c8c",
        "Gas Turbine": "#5f5f5f",
        "Hydro": "#4fc3f7",
        "Geothermal": "#8e44ad",
        "Biomass": "#2e7d32",
        "Waste": "#66bb6a",
        "Import": "#795548",
        "Oil": "#b71c1c",
        "Non-fossil": "#00acc1",
        "Other": "#9e9e9e",
    }

    present_resources = [r for r in resource_order if r in df["resource"].unique()]

    group_order = ["V1G", "V1G+V2G", "V2G"]
    participation_order = ["30%", "50%"]

    # Track which resources appear but are too small to label
    resources_in_legend = set()

    # Create subplots - one for each RPS value
    fig, axes = plt.subplots(1, n_rps, figsize=(6 * n_rps, 8), sharey=False, facecolor="white")

    # Handle case where n_rps = 1
    if n_rps == 1:
        axes = [axes]

    for spine_ax in axes:
        spine_ax.set_facecolor("white")
        for spine in spine_ax.spines.values():
            spine.set_color("black")
        spine_ax.tick_params(axis="both", colors="black")

    width = 0.35
    gap_between_groups = 0.5
    gap_between_participation = 0.1

    for rps_idx, rps in enumerate(rps_values):
        ax = axes[rps_idx]

        # Build ordered keys for this RPS
        ordered_keys = []
        for group in group_order:
            for participation in participation_order:
                ordered_keys.append((group, participation))

        # Build x positions for this subplot
        x_map = {}
        xticks = []
        xticklabels = []
        group_centers = {}
        participation_centers = {}

        current_x = 0
        for g_idx, group in enumerate(group_order):
            group_positions = []

            for p_idx, participation in enumerate(participation_order):
                x = current_x
                x_map[(group, participation)] = x
                xticks.append(x)
                xticklabels.append(f"{group}\n{participation}")
                group_positions.append(x)
                current_x += width + gap_between_participation

            participation_centers[group] = group_positions
            group_centers[group] = np.mean(group_positions)
            current_x += gap_between_groups - gap_between_participation

        # Vertical separator positions (between groups)
        separator_positions = []
        for g_idx in range(len(group_order) - 1):
            last_participation = participation_order[-1]
            last_group = group_order[g_idx]
            last_x = x_map[(last_group, last_participation)]
            next_participation = participation_order[0]
            next_group = group_order[g_idx + 1]
            next_x = x_map[(next_group, next_participation)]
            separator_positions.append((last_x + next_x) / 2)

        pos_bottom = {k: 0.0 for k in ordered_keys}
        neg_bottom = {k: 0.0 for k in ordered_keys}

        # Plot bars for this RPS
        for resource in present_resources:
            for key in ordered_keys:
                group, participation = key
                x = x_map[key]

                temp = df[
                    (df["group"] == group) &
                    (df["participation"] == participation) &
                    (df["rps"] == rps) &
                    (df["resource"] == resource)
                ]

                val = temp["delta_gwh"].sum() if not temp.empty else 0.0

                if val >= 0:
                    bottom = pos_bottom[key]
                    ax.bar(
                        x, val, width=width, bottom=bottom,
                        color=colors[resource], edgecolor="none"
                    )
                    # Only label if large enough
                    if abs(val) >= label_threshold:
                        ax.text(
                            x, bottom + val / 2, resource,
                            ha="center", va="center",
                            fontsize=10, color="black",
                        )
                    else:
                        # Track small resources for legend
                        if abs(val) > 0:
                            resources_in_legend.add(resource)
                    pos_bottom[key] += val
                else:
                    bottom = neg_bottom[key]
                    ax.bar(
                        x, val, width=width, bottom=bottom,
                        color=colors[resource], edgecolor="none"
                    )
                    # Only label if large enough
                    if abs(val) >= label_threshold:
                        ax.text(
                            x, bottom + val / 2, resource,
                            ha="center", va="center",
                            fontsize=10, color="black",
                        )
                    else:
                        # Track small resources for legend
                        if abs(val) > 0:
                            resources_in_legend.add(resource)
                    neg_bottom[key] += val

        ax.axhline(0, color="black", linewidth=1.2)

        for xpos in separator_positions:
            ax.axvline(xpos, color="black", linestyle="--", linewidth=1.0)

        ax.set_xticks(xticks)
        ax.set_xticklabels([])  # Remove default labels

        ymin, ymax = ax.get_ylim()
        y_range = ymax - ymin

        # Add group labels
        y_text_group = ymin - 0.08 * y_range
        for group in group_order:
            ax.text(
                group_centers[group], y_text_group, group,
                ha="center", va="top",
                fontsize=16, fontweight="bold", color="black"
            )

        # Add participation labels
        y_text_participation = ymin - 0.16 * y_range
        for group in group_order:
            for p_idx, participation in enumerate(participation_order):
                x_pos = participation_centers[group][p_idx]
                color = "#1f77b4" if participation == "30%" else "#d62728"
                ax.text(
                    x_pos, y_text_participation, participation,
                    ha="center", va="top",
                    fontsize=16, fontweight="bold", color=color
                )

        ax.set_title(f"RPS {rps}%", fontsize=16, fontweight="bold", color="black")
        ax.set_ylabel("Change in generation (GWh)", fontsize=16, color="black")
        ax.tick_params(axis='y', labelsize=16)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(axis="y", color="gray", alpha=0.3)

    # Create legend for small resources (only once for the entire figure)
    if resources_in_legend:
        legend_elements = [
            Patch(facecolor=colors[resource], edgecolor='black', label=resource)
            for resource in sorted(resources_in_legend)
        ]

        legend = fig.legend(
            handles=legend_elements,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.05),
            ncol=min(len(legend_elements), 6),
            frameon=True,

            fontsize=16,
            title="Small contributors (not labeled on bars)"
        )
        legend.get_title().set_fontsize(15)

    fig.suptitle(
        f"Generation change relative to Base only, battery capex = ${batt_capex_target}/kWh\n",
        # f"Blue = 10% participation, Red = 30% participation",
        fontsize=16,
        y=0.98,
        color="black"
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.96])

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    return df, fig, axes


def generation_summary_from_solution_json(solution_json_path, graph, time_step_h=1.0):
    import json
    import numpy as np
    import pandas as pd

    with open(solution_json_path) as f:
        data = json.load(f)

    rows = []

    for node in data["nodes"]:
        region = node["id"]

        if region not in graph._node:
            continue

        for handle, a_sol in node["assets"].items():

            if handle not in graph._node[region]["assets"]:
                continue

            meta = graph._node[region]["assets"][handle]

            cls = meta.get("_class", "")
            typ = meta.get("type", "")
            fuel = meta.get("fuel", None) or typ or cls

            # -------------------------
            # 1. Skip real demand
            # -------------------------
            if cls == "Load" and typ == "load":
                continue

            # -------------------------
            # 2. Storage → use production only
            # -------------------------
            if cls == "Store":
                prod = np.array(a_sol.get("production", []), dtype=float)

                if prod.size == 0:
                    continue

                handle_lower = str(handle).lower()
                type_lower = str(meta.get("type", "")).lower()
                fuel_lower = str(meta.get("fuel", "")).lower()

                is_v2g = (
                    type_lower == "ev_v2g"
                    or fuel_lower == "ev_v2g"
                    or "ev_v2g" in handle_lower
                )

                fuel_name = "v2g_discharge" if is_v2g else "battery_discharge"

                energy_gwh = prod.sum() * time_step_h / 1e9

                rows.append({
                    "resource": map_fuel_for_plot_extended(fuel_name),
                    "energy_gwh": energy_gwh
                })

                continue  # IMPORTANT: avoid double counting

            # -------------------------
            # 3. All other generators (including solar/wind)
            # -------------------------
            net = np.array(a_sol.get("net", []), dtype=float)

            if net.size == 0:
                continue

            gen = np.clip(net, 0, None)

            energy_gwh = gen.sum() * time_step_h / 1e9

            rows.append({
                "resource": map_fuel_for_plot_extended(fuel),
                "energy_gwh": energy_gwh
            })

    if len(rows) == 0:
        raise ValueError(f"No generation data found in {solution_json_path}")

    df = pd.DataFrame(rows)
    df = df.groupby("resource", as_index=False)["energy_gwh"].sum()

    return df

def generation_summary_from_solution_csv(solution_csv_path, graph, time_step_h=1.0):
    """
    Read generation directly from *_solution.csv.

    Important:
        Producers and stores can use ::production.
        Solar and wind are often modeled as Load assets, so we need ::net.
    """

    df_sol = pd.read_csv(solution_csv_path)

    rows = []

    # Use all asset net columns
    net_cols = [
        c for c in df_sol.columns
        if c.endswith("::net")
    ]

    # Use production columns for stores if available
    production_cols = [
        c for c in df_sol.columns
        if c.endswith("::production")
    ]

    # -----------------------------
    # 1. Read generation from ::net
    # -----------------------------
    for col in net_cols:
        asset_key = col.replace("::net", "")

        if ":" not in asset_key:
            continue

        region = asset_key.split(":", 1)[0]
        handle = asset_key.split(":", 1)[1]

        if region not in graph._node:
            continue

        if handle not in graph._node[region]["assets"]:
            continue

        meta = graph._node[region]["assets"][handle]

        cls = meta.get("_class", "")
        typ = meta.get("type", "")
        fuel = meta.get("fuel", None) or typ or cls

        # Skip normal demand load
        if cls == "Load" and typ == "load":
            continue

        # Skip storage here. We handle storage with production below.
        if cls == "Store":
            continue

        net_w = pd.to_numeric(df_sol[col], errors="coerce").fillna(0.0)

        # Generation is positive net
        gen_w = np.clip(net_w, 0, None)

        energy_gwh = gen_w.sum() * time_step_h / 1e9

        rows.append({
            "resource": map_fuel_for_plot_extended(fuel),
            "energy_gwh": energy_gwh,
        })

    # -----------------------------
    # 2. Read storage discharge from ::production
    # -----------------------------
    for col in production_cols:
        asset_key = col.replace("::production", "")

        if ":" not in asset_key:
            continue

        region = asset_key.split(":", 1)[0]
        handle = asset_key.split(":", 1)[1]

        if region not in graph._node:
            continue

        if handle not in graph._node[region]["assets"]:
            continue

        meta = graph._node[region]["assets"][handle]
        cls = meta.get("_class", "")

        # Only use production columns for storage.
        # Other generators were already counted using ::net.
        if cls != "Store":
            continue

        handle_lower = str(handle).lower()
        type_lower = str(meta.get("type", "")).lower()
        fuel_lower = str(meta.get("fuel", "")).lower()

        is_v2g = (
            type_lower == "ev_v2g"
            or fuel_lower == "ev_v2g"
            or "ev_v2g" in handle_lower
        )

        fuel = "v2g_discharge" if is_v2g else "battery_discharge"

        production_w = pd.to_numeric(df_sol[col], errors="coerce").fillna(0.0)
        energy_gwh = production_w.sum() * time_step_h / 1e9

        rows.append({
            "resource": map_fuel_for_plot_extended(fuel),
            "energy_gwh": energy_gwh,
        })

    if len(rows) == 0:
        raise ValueError(f"No generation columns found in {solution_csv_path}")

    gen = pd.DataFrame(rows)
    gen = gen.groupby("resource", as_index=False)["energy_gwh"].sum()

    return gen

#
# df_gen_delta_mid, fig, ax = plot_generation_delta_one_plot_two_sides(
#     state="CA",
#     adoption="slow",
#     graph_json_path=graph_json_path_target,
#     batt_capex_target=250,
#     rps_values=[50, 60, 70],
#     baseline_scenario_label="midnight",
#     program_scenario_label="flex",
#     save_path="figures/generation_delta_CA_mid_flex_vs_midnight.png",
# )

df_gen_delta_mid, fig, ax = plot_generation_delta_one_plot_two_sides(
    state="CA",
    adoption="fast",
    graph_json_path=graph_json_path_target,
    batt_capex_target=250,
    rps_values=[50, 60, 70],
    baseline_scenario_label="midnight",
    program_scenario_label="flex",
    save_path="figures/generation_delta_CA_mid_flex_vs_midnight.png",
)

df_gen_delta_mid, fig, ax = plot_generation_delta_one_plot_two_sides(
    state="CA",
    adoption="fast",
    graph_json_path=graph_json_path_target,
    batt_capex_target=250,
    rps_values=[50, 60, 70],
    baseline_scenario_label="midnight",
    program_scenario_label="flex",
    save_path="figures/generation_delta_CA_mid_flex_vs_midnight.png",
)

In [ ]:
def collect_estimated_curtailment_from_wastage(
    results_dirs,
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=30,
):
    if isinstance(results_dirs, str):
        results_dirs = [results_dirs]

    graph = good.graph.graph_from_json(graph_json_path)
    rows = []

    for results_dir in results_dirs:
        adoption = infer_adoption_from_results_dir(results_dir)
        folders = sorted(glob.glob(os.path.join(results_dir, "s*")))

        print(f"\nProcessing adoption scenario: {adoption} | {results_dir}")
        print(f"Found {len(folders)} scenario folders")

        for folder in folders:
            folder_name = os.path.basename(folder)
            meta = parse_scenario_metadata(folder_name)

            if meta["batt_capex"] != batt_capex_target:
                continue

            solution_csvs = glob.glob(os.path.join(folder, "*_solution.csv"))
            if not solution_csvs:
                continue

            try:
                solution = load_solution_as_object(solution_csvs[0])
                time_step_s, T = estimate_time_step_and_steps(solution, days=days)

                solar_curtail_gwh = 0.0
                wind_curtail_gwh = 0.0
                total_wastage_gwh = 0.0

                for region, node in solution._node.items():
                    if region not in graph._node:
                        continue

                    wastage = np.asarray(node.get("wastage", [0] * T), dtype=float).ravel()
                    # FIX: wastage is in Watts (power), multiply by time_step to get energy in Watt-seconds
                    # Then convert W·s to GWh: / 3.6e12
                    # OR if wastage is already energy (Joules), just divide by 3.6e12
                    # Based on your comment that results should be ~20,000 GWh, the current value of 0.02 is 1,000,000x too small
                    # This suggests we need to multiply by time_step_s first
                    total_wastage_gwh += (wastage.sum() * time_step_s) / 3.6e12

                    solar_available = np.zeros(T, dtype=float)
                    wind_available = np.zeros(T, dtype=float)

                    for handle, a_sol in node["assets"].items():
                        if handle not in graph._node[region]["assets"]:
                            continue

                        meta_asset = graph._node[region]["assets"][handle]
                        fuel = str(meta_asset.get("fuel", "")).lower()
                        typ = str(meta_asset.get("type", "")).lower()

                        label = fuel if fuel else typ
                        if label not in ["solar", "wind"]:
                            continue

                        profile = np.asarray(a_sol.get("profile", [0] * T), dtype=float).ravel()

                        capex_raw = a_sol.get("capex", 0)
                        if isinstance(capex_raw, list):
                            capex_val = float(capex_raw[0]) if len(capex_raw) > 0 else 0.0
                        else:
                            capex_val = float(capex_raw)

                        installed_capacity = float(meta_asset.get("installed_capacity", 0.0))
                        capacity = installed_capacity + capex_val

                        if capacity < 0:
                            available_gen = np.maximum(-(profile * capacity), 0.0)
                        else:
                            available_gen = np.maximum(profile * capacity, 0.0)

                        if label == "solar":
                            solar_available += available_gen
                        elif label == "wind":
                            wind_available += available_gen

                    renewable_available = solar_available + wind_available

                    solar_share = np.divide(
                        solar_available,
                        renewable_available,
                        out=np.zeros_like(solar_available),
                        where=renewable_available > 0
                    )

                    wind_share = np.divide(
                        wind_available,
                        renewable_available,
                        out=np.zeros_like(wind_available),
                        where=renewable_available > 0
                    )

                    solar_curtail = wastage * solar_share
                    wind_curtail = wastage * wind_share

                    # FIX: Apply time_step_s conversion here too
                    solar_curtail_gwh += (solar_curtail.sum() * time_step_s) / 3.6e12
                    wind_curtail_gwh += (wind_curtail.sum() * time_step_s) / 3.6e12

                rows.append({
                    "adoption": adoption,
                    "scenario_id": meta["scenario_id"],
                    "group": meta["group"],
                    "participation": meta["participation"],
                    "rps": meta["rps"],
                    "batt_capex": meta["batt_capex"],
                    "solar_curtailment_gwh": solar_curtail_gwh,
                    "wind_curtailment_gwh": wind_curtail_gwh,
                    "wastage_gwh": total_wastage_gwh,
                })

            except Exception as e:
                print(f"Could not process {folder}: {e}")

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("No curtailment rows were collected.")

    df["estimated_solar_wind_curtailment_gwh"] = (
        df["solar_curtailment_gwh"] + df["wind_curtailment_gwh"]
    )

    adoption_order = ["slow", "mid", "fast"]
    df["adoption"] = pd.Categorical(df["adoption"], categories=adoption_order, ordered=True)

    df = df.sort_values(["adoption", "group", "rps"]).reset_index(drop=True)
    return df

def collect_curtailment_deltas(
    results_dirs,
    graph_json_path,
    batt_capex_target=150,
    days=7
):
    dfc = collect_estimated_curtailment_from_wastage(
        results_dirs=results_dirs,
        graph_json_path=graph_json_path,
        batt_capex_target=batt_capex_target,
        days=days
    ).copy()

    dfc["resource"] = "Curtailment"
    dfc["energy_gwh"] = dfc["estimated_solar_wind_curtailment_gwh"]

    # Extract Base only data for comparison
    base = (
        dfc[dfc["group"] == "Base only"]
        .rename(columns={"energy_gwh": "base_energy_gwh"})
        [["adoption", "rps", "batt_capex", "resource", "base_energy_gwh"]]
    )

    # Get comparison data (V1G, V2G, V1G+V2G only)
    comp = dfc[dfc["group"].isin(["V1G", "V2G", "V1G+V2G"])].copy()

    # Merge to get deltas
    merged = comp.merge(
        base,
        on=["adoption", "rps", "batt_capex", "resource"],
        how="left"
    )

    merged["base_energy_gwh"] = merged["base_energy_gwh"].fillna(0.0)
    merged["delta_gwh"] = merged["energy_gwh"] - merged["base_energy_gwh"]

    return merged[["adoption", "group", "participation", "rps", "batt_capex", "resource", "energy_gwh", "base_energy_gwh", "delta_gwh"]]

def estimate_time_step_and_steps(solution, days):
    any_node = next(iter(solution._node.values()))
    T = len(any_node.get("shortfall", []))
    if T == 0:
        raise ValueError("Could not infer T from solution.")
    time_step_s = days * 24 * 3600 / T
    return time_step_s, T


def fuel_energy_capacity_summary(solution, graph, model, power_unit="W"):
    """
    Returns a table by fuel/type with:
        - total energy (GWh)
        - installed energy (GWh)
        - capex energy (GWh)
        - avg GW
        - installed capacity (GW)
        - expansion capacity built (GW)
        - total capacity (GW)
        - implied capacity factor
    """
    dt_h = float(model.time_step) / 3600.0
    T = len(model.steps)

    if power_unit.upper() == "W":
        energy_denom = 1e9   # W*h -> GWh
        power_denom  = 1e9   # W -> GW
    elif power_unit.upper() == "MW":
        energy_denom = 1e3   # MW*h -> GWh
        power_denom  = 1e3   # MW -> GW
    else:
        raise ValueError("power_unit must be 'W' or 'MW'")

    rows = []

    for region, node in solution._node.items():
        for handle, a_sol in node["assets"].items():

            if handle not in graph._node[region]["assets"]:
                continue

            meta = graph._node[region]["assets"][handle]
            cls  = meta.get("_class", "")
            typ  = meta.get("type", "")
            fuel = meta.get("fuel", None) or typ or cls

            is_gen = (cls == "Producer") or (cls == "Load" and typ != "load")
            if not is_gen:
                continue

            net = np.asarray(a_sol.get("net", [0] * T), dtype=float)

            capex_val = a_sol.get("capex", 0.0)
            if isinstance(capex_val, list):
                capex_val = float(capex_val[0]) if capex_val else 0.0
            capex_val = float(capex_val)

            installed = float(meta.get("installed_capacity", 0.0))
            total_cap = installed + capex_val

            profile = meta.get("profile", None)
            if isinstance(profile, str):
                profile = graph._node[region].get("profiles", {}).get(profile, None)

            if profile is None:
                prof = np.ones(T)
            else:
                prof = np.asarray(profile, dtype=float)[:T]

            avail_inst = installed * prof
            avail_cap  = capex_val * prof
            avail_tot  = avail_inst + avail_cap

            w_inst = np.zeros(T)
            mask = avail_tot > 0
            w_inst[mask] = avail_inst[mask] / avail_tot[mask]

            energy_total = net.sum() * dt_h / energy_denom
            energy_inst  = (net * w_inst).sum() * dt_h / energy_denom
            energy_cap   = energy_total - energy_inst

            avg_gw = net.mean() / power_denom

            rows.append({
                "fuel": fuel,
                "energy_gwh_total": energy_total,
                "energy_gwh_installed": energy_inst,
                "energy_gwh_capex": energy_cap,
                "avg_gw": avg_gw,
                "installed_capacity_gw": installed / power_denom,
                "capex_capacity_gw": capex_val / power_denom,
                "total_capacity_gw": total_cap / power_denom,
            })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df = df.groupby("fuel", as_index=False).sum()

    hours = T * dt_h
    df["capacity_factor"] = np.where(
        df["total_capacity_gw"] > 0,
        df["energy_gwh_total"] / (df["total_capacity_gw"] * hours),
        np.nan
    )

    df = df.sort_values("energy_gwh_total", ascending=False)
    return df


def make_simple_model_from_solution(solution, days):
    time_step_s, T = estimate_time_step_and_steps(solution, days=days)
    model = SimpleNamespace()
    model.time_step = time_step_s
    model.steps = list(range(T))
    return model


def infer_adoption_from_results_dir(results_dir):
    """
    Infer adoption scenario from folder name.
    Examples:
        scenario_results_2030_slow_even -> slow
        scenario_results_2030_mid_even  -> mid
        scenario_results_2030_fast_even -> fast
    """
    name = os.path.basename(os.path.normpath(results_dir)).lower()

    if "slow" in name:
        return "slow"
    elif "mid" in name:
        return "mid"
    elif "fast" in name:
        return "fast"
    else:
        return "unknown"

In [ ]:
def plot_curtailment_delta_grid(
    results_dirs,
    graph_json_path,
    batt_capex_targets=[150, 250],
    days=30,
    rps_values=[50, 60, 70],
    save_path=None
):
    """
    Create a 2x3 grid of curtailment plots:
    - Rows: Battery capex prices
    - Columns: RPS values
    - Lines: V1G, V2G, V1G+V2G (each with 10% and 30% participation)
    """
    import matplotlib.pyplot as plt
    import numpy as np

    # Collect data for all battery capex values
    all_dfs = []
    for batt_capex in batt_capex_targets:
        df = collect_curtailment_deltas(
            results_dirs=results_dirs,
            graph_json_path=graph_json_path,
            batt_capex_target=batt_capex,
            days=days
        ).copy()
        df['batt_capex'] = batt_capex
        all_dfs.append(df)

    df_all = pd.concat(all_dfs, ignore_index=True)

    # FIX: Don't include Base only since it's not in the delta data
    group_order = ["V1G", "V1G+V2G", "V2G"]
    participation_order = ["10%", "30%"]
    # FIX: Use lowercase to match the data
    adoption_order = ["slow", "mid", "fast"]

    # Line styles for participation
    linestyle_map = {
        "10%": "-",      # solid
        "30%": "--",     # dashed
    }

    # Colors for groups
    colors = {
        "V1G": "#4C78A8",
        "V2G": "#F28E2B",
        "V1G+V2G": "#72B7B2",
    }

    # Create subplots: rows = battery capex, cols = RPS
    n_rows = len(batt_capex_targets)
    n_cols = len(rps_values)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows),
                             sharex=True, sharey=False, facecolor="white")

    # Ensure axes is 2D array
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)

    for ax_row in axes.flatten():
        ax_row.set_facecolor("white")
        for spine in ax_row.spines.values():
            spine.set_color("black")
        ax_row.tick_params(axis="both", colors="black")

    # X-axis: adoption levels
    # FIX: Use lowercase keys
    x_positions = {"slow": 0, "mid": 1, "fast": 2}
    x_ticks = [0, 1, 2]
    # Display labels can still be capitalized
    x_labels = ["Slow", "Mid", "Fast"]

    # Plot each subplot
    for row_idx, batt_capex in enumerate(batt_capex_targets):
        for col_idx, rps in enumerate(rps_values):
            ax = axes[row_idx, col_idx]

            # Filter data for this subplot
            sub = df_all[(df_all['batt_capex'] == batt_capex) & (df_all['rps'] == rps)].copy()

            if sub.empty:
                ax.text(0.5, 0.5, "No data", ha="center", va="center",
                       transform=ax.transAxes, fontsize=12)
                ax.set_xticks(x_ticks)
                ax.set_xticklabels(x_labels)
                continue

            # Plot lines for each group × participation combination
            for group in group_order:
                for participation in participation_order:
                    # Get data for this combination
                    group_data = sub[(sub['group'] == group) &
                                    (sub['participation'] == participation)].copy()

                    if group_data.empty:
                        continue

                    # Extract values for each adoption level
                    x_vals = []
                    y_vals = []
                    for adoption in adoption_order:
                        adoption_data = group_data[group_data['adoption'] == adoption]
                        if not adoption_data.empty:
                            x_vals.append(x_positions[adoption])
                            y_vals.append(adoption_data['delta_gwh'].sum())

                    if x_vals:
                        # Create label
                        label = f"{group} {participation}"

                        # Plot line
                        ax.plot(
                            x_vals, y_vals,
                            marker='o',
                            linewidth=2,
                            linestyle=linestyle_map[participation],
                            color=colors[group],
                            label=label,
                            markersize=6
                        )

            # Formatting
            ax.axhline(0, color="black", linewidth=1, linestyle="-", alpha=0.5)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(x_labels)
            ax.grid(True, axis="y", alpha=0.3, linestyle="--")

            # Title for each subplot
            if row_idx == 0:
                ax.set_title(f"RPS {rps}%", fontsize=14, fontweight="bold")

            # Y-label for leftmost column
            if col_idx == 0:
                ax.set_ylabel(
                    f"Curtailment change (GWh)\nBattery ${batt_capex}/kWh",
                    fontsize=12
                )

            # X-label for bottom row
            if row_idx == n_rows - 1:
                ax.set_xlabel("Adoption scenario", fontsize=12)

    # Single legend for the entire figure
    handles, labels = axes[0, 0].get_legend_handles_labels()

    # Remove duplicates
    by_label = dict(zip(labels, handles))

    if by_label:  # Only add legend if there are items
        fig.legend(
            by_label.values(),
            by_label.keys(),
            loc="lower center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=3,
            frameon=True,
            fontsize=10,
            title="Scenarios (solid=10%, dashed=30%)"
        )

    fig.suptitle(
        f"Curtailment change relative to Base only\n"
        f"Over {days} days",
        fontsize=16,
        y=0.98,
        color="black"
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.96])

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return df_all, fig, axes


# Usage example
df_curtail, fig, axes = plot_curtailment_delta_grid(
    results_dirs=[
        f"scenario_results_2030_slow_even_{state}_flex",
        f"scenario_results_2030_mid_even_{state}_flex",
        f"scenario_results_2030_fast_even_{state}_flex",
    ],
    graph_json_path=graph_json_path_target,
    batt_capex_targets=[150, 250],
    days=7,
    rps_values=[50, 60, 70],
    save_path="figures/curtailment_delta_grid.png"
)

In [ ]:
def plot_solar_wind_generation_changes_3x3(
    df_dict,
    group_order=("V1G", "V2G", "V1G+V2G"),
    adoption_order=("Slow", "Mid", "Fast"),
    figsize=(15, 12),
    sharex=True,
    sharey="row",
    save_path=None
):

    import os
    import pandas as pd
    import matplotlib.pyplot as plt

    metric_names = ["Solar", "Wind", "Solar+Wind"]
    row_titles = ["Solar", "Wind", "Solar + Wind"]
    col_titles = {
        "Slow": "Slow adoption scenario",
        "Mid": "Mid adoption scenario",
        "Fast": "Fast adoption scenario"
    }

    color_map = {
        "V1G": "#3182bd",
        "V2G": "#31a354",
        "V1G+V2G": "#e6550d"
    }

    pivot_delta_dict = {}

    fig, axes = plt.subplots(
        nrows=3,
        ncols=len(adoption_order),
        figsize=figsize,
        sharex=sharex,
        sharey=sharey
        )
    fig.suptitle(
        "Change in Solar and Wind Generation Relative to Base Across Charging Strategies and Adoption Scenarios",
        fontsize=16
    )
    for j, adoption in enumerate(adoption_order):
        if adoption not in df_dict:
            continue

        df_gen_delta = df_dict[adoption].copy()

        df_sw = df_gen_delta[df_gen_delta["resource"].isin(["Solar", "Wind"])].copy()

        pivot_delta = df_sw.pivot_table(
            index=["group", "rps"],
            columns="resource",
            values="delta_gwh",
            aggfunc="sum"
        ).reset_index()

        pivot_delta = pivot_delta.fillna(0)

        if "Solar" not in pivot_delta.columns:
            pivot_delta["Solar"] = 0.0
        if "Wind" not in pivot_delta.columns:
            pivot_delta["Wind"] = 0.0

        pivot_delta["Solar+Wind"] = pivot_delta["Solar"] + pivot_delta["Wind"]

        existing_groups = [g for g in group_order if g in pivot_delta["group"].unique()]
        if existing_groups:
            pivot_delta["group"] = pd.Categorical(
                pivot_delta["group"], categories=existing_groups, ordered=True
            )
            pivot_delta = pivot_delta.sort_values(["group", "rps"]).reset_index(drop=True)

        pivot_delta_dict[adoption] = pivot_delta

        for i, metric in enumerate(metric_names):
            ax = axes[i, j]

            line_end_info = []

            for g in existing_groups:
                temp = pivot_delta[pivot_delta["group"] == g].copy()
                if temp.empty:
                    continue

                temp = temp.sort_values("rps")

                ax.plot(
                    temp["rps"],
                    temp[metric],
                    marker="o",
                    linewidth=2,
                    color=color_map.get(g, None)
                )

                # store last point for direct labeling
                x_last = temp["rps"].iloc[-1]
                y_last = temp[metric].iloc[-1]
                line_end_info.append((g, x_last, y_last))

            ax.axhline(0, color="black", linestyle="--", linewidth=1)
            ax.grid(alpha=0.3)

            if i == 0:
                ax.set_title(col_titles.get(adoption, adoption), fontsize=13)

            if j == 0:
                ax.set_ylabel(f"{row_titles[i]}\nΔ Generation (GWh)", fontsize=11)

            if i == 2:
                ax.set_xlabel("RPS (%)", fontsize=11)

            # direct labels on right side of lines
            if line_end_info:
                line_end_info = sorted(line_end_info, key=lambda x: x[2])

                # small vertical offsets to reduce overlap
                offsets = [-12, 0, 12]
                if len(line_end_info) > len(offsets):
                    offsets = list(pd.Series(range(len(line_end_info))).map(lambda k: -12 + 24 * k / max(len(line_end_info)-1, 1)))

                for k, (g, x_last, y_last) in enumerate(line_end_info):
                    dy = offsets[k] if k < len(offsets) else 0
                    ax.annotate(
                        g,
                        xy=(x_last, y_last),
                        xytext=(8, dy),
                        textcoords="offset points",
                        ha="left",
                        va="center",
                        fontsize=9,
                        color=color_map.get(g, "black"),
                        fontweight="bold"
                    )

    # no legend
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")

    return {
        "pivot_delta_dict": pivot_delta_dict,
        "fig": fig,
        "axes": axes
    }

results_3x3 = plot_solar_wind_generation_changes_3x3(
    df_dict={
        "Slow": df_gen_delta_slow,
        "Mid": df_gen_delta_mid,
        "Fast": df_gen_delta_fast
    }
)

In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
from types import SimpleNamespace

def collect_renewable_build_summary(
    results_dirs,
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=7,
    target_fuel="solar"
):
    """
    Collect renewable build summary across multiple adoption scenario folders.

    Parameters
    ----------
    results_dirs : str or list of str
        Example:
        [
            "scenario_results_2030_slow_even",
            "scenario_results_2030_mid_even",
            "scenario_results_2030_fast_even",
        ]
    """
    if isinstance(results_dirs, str):
        results_dirs = [results_dirs]

    graph = good.graph.graph_from_json(graph_json_path)
    rows = []
    target_fuel = target_fuel.lower()

    for results_dir in results_dirs:
        adoption = infer_adoption_from_results_dir(results_dir)
        folders = sorted(glob.glob(os.path.join(results_dir, "s*")))

        print(f"\nProcessing adoption scenario: {adoption} | {results_dir}")
        print(f"Found {len(folders)} scenario folders")

        for folder in folders:
            folder_name = os.path.basename(folder)
            meta = parse_scenario_metadata(folder_name)

            if meta["batt_capex"] != batt_capex_target:
                continue

            solution_csvs = glob.glob(os.path.join(folder, "*_solution.csv"))
            if not solution_csvs:
                continue

            solution_json_path = solution_jsons[0]

            try:
                solution = load_solution_as_object(solution_json_path)
                model = make_simple_model_from_solution(solution, days=days)

                summary = fuel_energy_capacity_summary(
                    solution=solution,
                    graph=graph,
                    model=model,
                    power_unit="W"
                )

                if summary.empty:
                    continue

                fuel_rows = summary[
                    summary["fuel"].astype(str).str.lower() == target_fuel
                ].copy()

                print(
                    f"Scenario {meta['scenario_id']} | "
                    f"{adoption} | {meta['group']} | RPS {meta['rps']}"
                )

                if fuel_rows.empty:
                    print(f"{target_fuel.capitalize()} rows found: 0")
                    continue

                print(f"{target_fuel.capitalize()} rows found: {len(fuel_rows)}")

                fuel_row = fuel_rows.iloc[0]

                rows.append({
                    "adoption": adoption,
                    "scenario_id": meta["scenario_id"],
                    "group": meta["group"],
                    "rps": meta["rps"],
                    "batt_capex": meta["batt_capex"],
                    "resource": target_fuel.capitalize(),
                    "energy_gwh_total": fuel_row["energy_gwh_total"],
                    "capex_capacity_gw": fuel_row["capex_capacity_gw"],
                    "total_capacity_gw": fuel_row["total_capacity_gw"],
                })

            except Exception as e:
                print(f"Could not process {solution_json_path}: {e}")

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(
            f"No {target_fuel} build rows were collected. Check printed errors above."
        )

    adoption_order = ["slow", "mid", "fast"]
    if "adoption" in df.columns:
        df["adoption"] = pd.Categorical(df["adoption"], categories=adoption_order, ordered=True)

    df = df.sort_values(["adoption", "group", "rps"]).reset_index(drop=True)
    return df


df_solar_build = collect_renewable_build_summary(
    results_dirs=[
        f"scenario_results_2030_slow_even_{state}_flex",
        f"scenario_results_2030_mid_even_{state}_flex",
        f"scenario_results_2030_fast_even_{state}_flex",
    ],
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=7,
    target_fuel="solar"
)

df_wind_build = collect_renewable_build_summary(
    results_dirs=[
        f"scenario_results_2030_slow_even_{state}_flex",
        f"scenario_results_2030_mid_even_{state}_flex",
        f"scenario_results_2030_fast_even_{state}_flex",
    ],
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=7,
    target_fuel="wind"
)

In [ ]:

def infer_adoption_from_results_dir(results_dir):
    """
    Infer adoption scenario from folder name.
    """
    name = os.path.basename(os.path.normpath(results_dir)).lower()

    if "slow" in name:
        return "slow"
    elif "mid" in name:
        return "mid"
    elif "fast" in name:
        return "fast"
    else:
        return "unknown"


def collect_renewable_curtailment_summary(
    results_dirs,
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=30,
    target_fuel="solar"
):
    """
    Collect renewable curtailment summary across multiple adoption scenarios.

    Parameters
    ----------
    results_dirs : str or list of str
        Example:
        [
            "scenario_results_2030_slow_even",
            "scenario_results_2030_mid_even",
            "scenario_results_2030_fast_even",
        ]
    """
    if isinstance(results_dirs, str):
        results_dirs = [results_dirs]

    graph = good.graph.graph_from_json(graph_json_path)
    rows = []
    target_fuel = target_fuel.lower()

    for results_dir in results_dirs:
        adoption = infer_adoption_from_results_dir(results_dir)
        folders = sorted(glob.glob(os.path.join(results_dir, "s*")))

        print(f"\nProcessing adoption scenario: {adoption} | {results_dir}")
        print(f"Found {len(folders)} scenario folders")

        for folder in folders:
            folder_name = os.path.basename(folder)
            meta = parse_scenario_metadata(folder_name)

            if meta["batt_capex"] != batt_capex_target:
                continue

            solution_jsons = glob.glob(os.path.join(folder, "*_solution.json"))
            if not solution_jsons:
                continue

            solution_json_path = solution_jsons[0]

            try:
                solution = load_solution_as_object(solution_json_path)
                time_step_s, T = estimate_time_step_and_steps(solution, days=days)
                dt_h = time_step_s / 3600.0

                fuel_curtail_gwh = 0.0
                fuel_gen_gwh = 0.0

                for region, node in solution._node.items():
                    for handle, a_sol in node["assets"].items():

                        if handle not in graph._node[region]["assets"]:
                            continue

                        meta_asset = graph._node[region]["assets"][handle]
                        fuel = str(meta_asset.get("fuel", "")).lower()

                        if fuel != target_fuel:
                            continue

                        curtail = np.asarray(a_sol.get("curtail", [0] * T), dtype=float).ravel()
                        net = np.asarray(a_sol.get("net", [0] * T), dtype=float).ravel()

                        fuel_curtail_gwh += curtail.sum() * dt_h / 1e9
                        fuel_gen_gwh += np.clip(net, 0, None).sum() * dt_h / 1e9

                rows.append({
                    "adoption": adoption,
                    "scenario_id": meta["scenario_id"],
                    "group": meta["group"],
                    "rps": meta["rps"],
                    "batt_capex": meta["batt_capex"],
                    "resource": target_fuel.capitalize(),
                    f"{target_fuel}_generation_gwh": fuel_gen_gwh,
                    f"{target_fuel}_curtailment_gwh": fuel_curtail_gwh,
                })

            except Exception as e:
                print(f"Could not process {solution_json_path}: {e}")

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(f"No {target_fuel} curtailment rows were collected.")

    gen_col = f"{target_fuel}_generation_gwh"
    curt_col = f"{target_fuel}_curtailment_gwh"
    pot_col = f"{target_fuel}_potential_gwh"
    rate_col = f"{target_fuel}_curtailment_rate"

    df[pot_col] = df[gen_col] + df[curt_col]
    df[rate_col] = np.where(
        df[pot_col] > 0,
        df[curt_col] / df[pot_col],
        np.nan
    )

    adoption_order = ["slow", "mid", "fast"]
    df["adoption"] = pd.Categorical(df["adoption"], categories=adoption_order, ordered=True)

    df = df.sort_values(["adoption", "group", "rps"]).reset_index(drop=True)
    return df


# -------------------------------------------------------------------
# Collect solar and wind curtailment for all three adoption scenarios
# -------------------------------------------------------------------

results_dirs = [
        f"scenario_results_2030_slow_even_{state}_flex",
        f"scenario_results_2030_mid_even_{state}_flex",
        f"scenario_results_2030_fast_even_{state}_flex",
]

df_solar_curt = collect_renewable_curtailment_summary(
    results_dirs=results_dirs,
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=30,
    target_fuel="solar"
)

df_wind_curt = collect_renewable_curtailment_summary(
    results_dirs=results_dirs,
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=30,
    target_fuel="wind"
)

print(df_solar_curt)
print(df_wind_curt)


# -------------------------------------------------------------------
# Pivot tables
# -------------------------------------------------------------------

group_order = ["Base only", "V1G", "V2G", "V1G+V2G"]


solar_build_table = df_solar_build.pivot_table(
    index=["adoption", "rps"],
    columns="group",
    values=["energy_gwh_total", "capex_capacity_gw", "total_capacity_gw"],
    aggfunc="first"
)
solar_build_table = solar_build_table.reindex(columns=group_order, level=1)
print(solar_build_table.round(3))


solar_curt_table = df_solar_curt.pivot_table(
    index=["adoption", "rps"],
    columns="group",
    values=["solar_generation_gwh", "solar_curtailment_gwh", "solar_curtailment_rate"],
    aggfunc="first"
)
solar_curt_table = solar_curt_table.reindex(columns=group_order, level=1)
print(solar_curt_table.round(3))


wind_build_table = df_wind_build.pivot_table(
    index=["adoption", "rps"],
    columns="group",
    values=["energy_gwh_total", "capex_capacity_gw", "total_capacity_gw"],
    aggfunc="first"
)
wind_build_table = wind_build_table.reindex(columns=group_order, level=1)
print(wind_build_table.round(3))


wind_curt_table = df_wind_curt.pivot_table(
    index=["adoption", "rps"],
    columns="group",
    values=["wind_generation_gwh", "wind_curtailment_gwh", "wind_curtailment_rate"],
    aggfunc="first"
)
wind_curt_table = wind_curt_table.reindex(columns=group_order, level=1)
print(wind_curt_table.round(3))

In [ ]:



def infer_adoption_from_results_dir(results_dir):
    name = os.path.basename(os.path.normpath(results_dir)).lower()
    if "slow" in name:
        return "slow"
    elif "mid" in name:
        return "mid"
    elif "fast" in name:
        return "fast"
    return "unknown"


def collect_total_demand_summary(
    results_dirs,
    batt_capex_target=150,
    days=30
):
    """
    Collect total demand directly from solution assets.

    This uses absolute value of net for load assets because the saved
    solution may store base load and EV load with different sign conventions.

    Returns
    -------
    DataFrame with:
        adoption
        scenario_id
        group
        rps
        batt_capex
        base_demand_gwh
        ev_demand_gwh
        total_demand_gwh
    """
    if isinstance(results_dirs, str):
        results_dirs = [results_dirs]

    rows = []

    for results_dir in results_dirs:
        adoption = infer_adoption_from_results_dir(results_dir)
        folders = sorted(glob.glob(os.path.join(results_dir, "s*")))

        print(f"\nProcessing total demand for {adoption} adoption")

        for folder in folders:
            folder_name = os.path.basename(folder)
            meta = parse_scenario_metadata(folder_name)

            if meta["batt_capex"] != batt_capex_target:
                continue

            solution_jsons = glob.glob(os.path.join(folder, "*_solution.json"))
            if not solution_jsons:
                continue

            solution_json_path = solution_jsons[0]

            try:
                solution = load_solution_as_object(solution_json_path)
                time_step_s, T = estimate_time_step_and_steps(solution, days=days)
                dt_h = time_step_s / 3600.0

                base_demand_gwh = 0.0
                ev_demand_gwh = 0.0

                for region, node in solution._node.items():
                    for handle, a_sol in node["assets"].items():
                        handle_str = str(handle).lower()
                        net = np.asarray(a_sol.get("net", [0] * T), dtype=float).ravel()

                        # demand magnitude
                        demand_gwh = np.maximum(-net, 0).sum() * dt_h / 1e9

                        if handle_str.startswith("ev_load_"):
                            demand_gwh = np.maximum(-net, 0).sum() * dt_h / 1e9
                            ev_demand_gwh += demand_gwh

                        elif "load" in handle_str and not handle_str.startswith("ev_"):
                            demand_gwh = np.maximum(-net, 0).sum() * dt_h / 1e9
                            base_demand_gwh += demand_gwh

                total_demand_gwh = base_demand_gwh + ev_demand_gwh

                rows.append({
                    "adoption": adoption,
                    "scenario_id": meta["scenario_id"],
                    "group": meta["group"],
                    "rps": meta["rps"],
                    "batt_capex": meta["batt_capex"],
                    "base_demand_gwh": base_demand_gwh,
                    "ev_demand_gwh": ev_demand_gwh,
                    "total_demand_gwh": total_demand_gwh,
                })

            except Exception as e:
                print(f"Could not process {solution_json_path}: {e}")

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("No demand rows were collected.")

    adoption_order = ["slow", "mid", "fast"]
    df["adoption"] = pd.Categorical(df["adoption"], categories=adoption_order, ordered=True)
    df = df.sort_values(["adoption", "group", "rps"]).reset_index(drop=True)

    return df

df_total_demand = collect_total_demand_summary(
    results_dirs=[
        f"scenario_results_2030_slow_even_{state}_flex",
        f"scenario_results_2030_mid_even_{state}_flex",
        f"scenario_results_2030_fast_even_{state}_flex",
    ],
    batt_capex_target=150,
    days=30
)


In [ ]:
adoption_order = ["slow", "mid", "fast"]
group_order = ["Base only", "V1G", "V2G", "V1G+V2G"]

plot_specs = [
    {
        "df": df_total_demand,
        "metric": "total_demand_gwh",
        "row_title": "Total demand",
        "ylabel": "GWh"
    },
    {
        "df": df_solar_build,
        "metric": "energy_gwh_total",
        "row_title": "Solar generation",
        "ylabel": "GWh"
    },
    {
        "df": df_wind_build,
        "metric": "energy_gwh_total",
        "row_title": "Wind generation",
        "ylabel": "GWh"
    },
    {
        "df": df_solar_build,
        "metric": "capex_capacity_gw",
        "row_title": "Solar capex",
        "ylabel": "GW"
    },
    {
        "df": df_wind_build,
        "metric": "capex_capacity_gw",
        "row_title": "Wind capex",
        "ylabel": "GW"
    },
]

fig, axes = plt.subplots(
    nrows=len(plot_specs),
    ncols=3,
    figsize=(16, 18),
    sharex=False,
    sharey="row"
)

for row_i, spec in enumerate(plot_specs):
    df_plot = spec["df"]
    metric = spec["metric"]
    row_title = spec["row_title"]
    ylabel = spec["ylabel"]

    for col_i, adoption in enumerate(adoption_order):
        ax = axes[row_i, col_i]
        sub = df_plot[df_plot["adoption"] == adoption].copy()

        if sub.empty:
            ax.set_title(f"{adoption.capitalize()} adoption")
            ax.set_xlabel("RPS")
            ax.set_ylabel(ylabel)
            ax.grid(True, linestyle="--", alpha=0.4)
            continue

        pivot = sub.pivot(index="rps", columns="group", values=metric)

        existing_cols = [g for g in group_order if g in pivot.columns]
        pivot = pivot.reindex(columns=existing_cols)

        # Plot lines WITHOUT legend
        lines = []
        for col in pivot.columns:
            line, = ax.plot(pivot.index, pivot[col], marker="o", label=col)
            lines.append((line, col))

        # -----------------------
        # Ladder-style labeling
        # -----------------------
        if row_i != 0:
            x = pivot.index.values
            if len(x) > 0:
                x_text = x[-1] + 0.2

                # collect end points
                label_data = []
                for line, label in lines:
                    y = pivot[label].values
                    if len(y) == 0:
                        continue
                    label_data.append([label, y[-1]])

                # sort by final y value
                label_data.sort(key=lambda z: z[1])

                # enforce a minimum vertical gap
                y_min, y_max = ax.get_ylim()
                min_gap = 0.03 * (y_max - y_min)

                adjusted_y = []
                for j, (label, y_end) in enumerate(label_data):
                    if j == 0:
                        adjusted_y.append(y_end)
                    else:
                        adjusted_y.append(max(y_end, adjusted_y[-1] + min_gap))

                # draw labels
                for (label, y_end), y_lab in zip(label_data, adjusted_y):
                    ax.text(
                        x_text,
                        y_lab,
                        label,
                        fontsize=10,
                        va="center"
                    )

        # Titles
        if row_i == 0:
            ax.set_title(f"{adoption.capitalize()} adoption")
        else:
            ax.set_title("")

        ax.set_xlabel("RPS")
        ax.set_ylabel(ylabel)

        # Row labels
        if col_i == 0:
            ax.text(
                -0.18, 0.5, row_title,
                transform=ax.transAxes,
                rotation=90,
                va="center",
                ha="center",
                fontsize=12,
                fontweight="bold"
            )

        # -----------------------
        # Grid
        # -----------------------
        ax.grid(True, linestyle="--", alpha=0.4)

        # remove legend if any
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # extend x-limit so labels fit
        ax.set_xlim(45, 85)

# Title
fig.suptitle(
    "Demand, generation, and capacity expansion across adoption scenarios",
    fontsize=16,
    y=0.995
)

fig.tight_layout(rect=[0.04, 0.03, 1, 0.985])
plt.show()

In [ ]:
def collect_estimated_curtailment_from_wastage(
    results_dirs,
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=30,
):
    if isinstance(results_dirs, str):
        results_dirs = [results_dirs]

    graph = good.graph.graph_from_json(graph_json_path)
    rows = []

    for results_dir in results_dirs:
        adoption = infer_adoption_from_results_dir(results_dir)
        folders = sorted(glob.glob(os.path.join(results_dir, "s*")))

        print(f"\nProcessing adoption scenario: {adoption} | {results_dir}")
        print(f"Found {len(folders)} scenario folders")

        for folder in folders:
            folder_name = os.path.basename(folder)
            meta = parse_scenario_metadata(folder_name)

            if meta["batt_capex"] != batt_capex_target:
                continue

            solution_jsons = glob.glob(os.path.join(folder, "*_solution.json"))
            if not solution_jsons:
                continue

            try:
                solution = load_solution_as_object(solution_jsons[0])
                time_step_s, T = estimate_time_step_and_steps(solution, days=days)

                solar_curtail_gwh = 0.0
                wind_curtail_gwh = 0.0
                total_wastage_gwh = 0.0

                for region, node in solution._node.items():
                    if region not in graph._node:
                        continue

                    wastage = np.asarray(node.get("wastage", [0] * T), dtype=float).ravel()
                    total_wastage_gwh += wastage.sum() / 3.6e12

                    solar_available = np.zeros(T, dtype=float)
                    wind_available = np.zeros(T, dtype=float)

                    for handle, a_sol in node["assets"].items():
                        if handle not in graph._node[region]["assets"]:
                            continue

                        meta_asset = graph._node[region]["assets"][handle]
                        fuel = str(meta_asset.get("fuel", "")).lower()
                        typ = str(meta_asset.get("type", "")).lower()

                        label = fuel if fuel else typ
                        if label not in ["solar", "wind"]:
                            continue

                        profile = np.asarray(a_sol.get("profile", [0] * T), dtype=float).ravel()

                        capex_raw = a_sol.get("capex", 0)
                        if isinstance(capex_raw, list):
                            capex_val = float(capex_raw[0]) if len(capex_raw) > 0 else 0.0
                        else:
                            capex_val = float(capex_raw)

                        installed_capacity = float(meta_asset.get("installed_capacity", 0.0))
                        capacity = installed_capacity + capex_val

                        if capacity < 0:
                            available_gen = np.maximum(-(profile * capacity), 0.0)
                        else:
                            available_gen = np.maximum(profile * capacity, 0.0)

                        if label == "solar":
                            solar_available += available_gen
                        elif label == "wind":
                            wind_available += available_gen

                    renewable_available = solar_available + wind_available

                    solar_share = np.divide(
                        solar_available,
                        renewable_available,
                        out=np.zeros_like(solar_available),
                        where=renewable_available > 0
                    )

                    wind_share = np.divide(
                        wind_available,
                        renewable_available,
                        out=np.zeros_like(wind_available),
                        where=renewable_available > 0
                    )

                    solar_curtail = wastage * solar_share
                    wind_curtail = wastage * wind_share

                    solar_curtail_gwh += solar_curtail.sum() / 3.6e12
                    wind_curtail_gwh += wind_curtail.sum() / 3.6e12

                rows.append({
                    "adoption": adoption,
                    "scenario_id": meta["scenario_id"],
                    "group": meta["group"],
                    "rps": meta["rps"],
                    "batt_capex": meta["batt_capex"],
                    "solar_curtailment_gwh": solar_curtail_gwh,
                    "wind_curtailment_gwh": wind_curtail_gwh,
                    "wastage_gwh": total_wastage_gwh,
                })

            except Exception as e:
                print(f"Could not process {folder}: {e}")

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("No curtailment rows were collected.")

    df["estimated_solar_wind_curtailment_gwh"] = (
        df["solar_curtailment_gwh"] + df["wind_curtailment_gwh"]
    )

    adoption_order = ["slow", "mid", "fast"]
    df["adoption"] = pd.Categorical(df["adoption"], categories=adoption_order, ordered=True)

    df = df.sort_values(["adoption", "group", "rps"]).reset_index(drop=True)
    return df

In [ ]:
import matplotlib.pyplot as plt

group_order = ["Base only", "V1G", "V2G", "V1G+V2G"]
adoption_order = ["slow", "mid", "fast"]

df_curt = collect_estimated_curtailment_from_wastage(
    results_dirs=[
        f"scenario_results_2030_slow_even_{state}_flex",
        f"scenario_results_2030_mid_even_{state}_flex",
        f"scenario_results_2030_fast_even_{state}_flex",
    ],
    graph_json_path=graph_json_path_target,
    batt_capex_target=150,
    days=30,
)


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
fig.subplots_adjust(top=0.92, hspace=0.25)
fig.text(0.5, 0.97, "Total Node Level Wastage (GWh)", ha="center", fontsize=14, fontweight="bold")
fig.text(0.5, 0.48, "Change in Wastage Relative to Base (GWh)", ha="center", fontsize=14, fontweight="bold")

group_colors = {
    "Base only": "tab:blue",
    "V1G":       "tab:orange",
    "V2G":       "tab:green",
    "V1G+V2G":   "tab:red",
}

for i, adoption in enumerate(adoption_order):
    ax_top = axes[0, i]
    ax_bottom = axes[1, i]

    sub = df_curt[df_curt["adoption"] == adoption]

    pivot = sub.pivot_table(
        index="rps",
        columns="group",
        values="wastage_gwh",
        aggfunc="sum"
    ).sort_index()

    existing_cols = [g for g in group_order if g in pivot.columns]
    pivot = pivot.reindex(columns=existing_cols)

    # Top row: absolute wastage
    for col in pivot.columns:
        ax_top.plot(pivot.index, pivot[col], marker="o",
                    color=group_colors.get(col), label=col)

    ax_top.set_title(f"{adoption.capitalize()} adoption")
    ax_top.grid(True, linestyle="--", alpha=0.4)

    # Bottom row: difference vs Base only
    if "Base only" in pivot.columns:
        delta = pivot.subtract(pivot["Base only"], axis=0)
        compare_cols = [g for g in ["V1G", "V2G", "V1G+V2G"] if g in delta.columns]

        for col in compare_cols:
            ax_bottom.plot(delta.index, delta[col], marker="o",
                           color=group_colors.get(col), label=col)

        ax_bottom.axhline(0, color="black", linestyle="--", linewidth=1)
        ax_bottom.grid(True, linestyle="--", alpha=0.4)
        ax_bottom.set_xlabel("RPS (%)")

axes[0, 0].set_ylabel("Wastage (GWh)")
axes[1, 0].set_ylabel("Δ Wastage vs Base (GWh)")

handles, labels = axes[0, -1].get_legend_handles_labels()
axes[0, -1].legend(handles, labels, loc="best")

plt.show()

In [ ]:
def check_ev_demand(solution_json_path, days=30):
    solution = load_solution_as_object(solution_json_path)
    time_step_s, T = estimate_time_step_and_steps(solution, days=days)
    dt_h = time_step_s / 3600.0

    ev_demand_gwh = 0.0

    for region, node in solution._node.items():
        for handle, a_sol in node["assets"].items():
            if str(handle).startswith("ev_load_"):
                net = np.asarray(a_sol.get("net", [0] * T), dtype=float).ravel()
                ev_demand_gwh += np.maximum(-net, 0).sum() * dt_h / 1e9

    return ev_demand_gwh

def check_ev_demand(solution_json_path, days=30):
    solution = load_solution_as_object(solution_json_path)
    time_step_s, T = estimate_time_step_and_steps(solution, days=days)
    dt_h = time_step_s / 3600.0

    ev_demand_gwh = 0.0

    for region, node in solution._node.items():
        for handle, a_sol in node["assets"].items():
            if str(handle).startswith("ev_load_"):
                net = np.asarray(a_sol.get("net", [0] * T), dtype=float).ravel()
                ev_demand_gwh += np.maximum(-net, 0).sum() * dt_h / 1e9

    return ev_demand_gwh

def check_positive_v1g_hours(solution_json_path):
    solution = load_solution_as_object(solution_json_path)

    for region, node in solution._node.items():
        for handle, a_sol in node["assets"].items():
            if str(handle).startswith("ev_load_v1g_"):
                net = np.asarray(a_sol.get("net", []), dtype=float)
                n_pos = np.sum(net > 1e-6)
                max_pos = net.max() if len(net) else 0
                print(region, handle, "positive_hours =", n_pos, "max_positive =", max_pos)


def check_v1g_shift_sum(solution_json_path):
    solution = load_solution_as_object(solution_json_path)

    for region, node in solution._node.items():
        for handle, a_sol in node["assets"].items():
            if str(handle).startswith("ev_load_v1g_"):
                shift = np.asarray(a_sol.get("shift", []), dtype=float)
                print(region, handle, "total shift sum =", shift.sum())

def check_wastage_and_imports(solution_json_path, graph_json_path="Examples/California_new.json", days=30):
    solution = load_solution_as_object(solution_json_path)
    graph = good.graph.graph_from_json(graph_json_path)

    time_step_s, T = estimate_time_step_and_steps(solution, days=days)
    dt_h = time_step_s / 3600.0

    total_wastage_gwh = 0.0
    total_import_gwh = 0.0

    for region, node in solution._node.items():
        wastage = np.asarray(node.get("wastage", [0] * T), dtype=float).ravel()
        total_wastage_gwh += wastage.sum() * dt_h / 1e9

        for handle, a_sol in node["assets"].items():
            if handle not in graph._node[region]["assets"]:
                continue
            meta_asset = graph._node[region]["assets"][handle]
            fuel = str(meta_asset.get("fuel", "")).lower()

            if fuel == "import":
                net = np.asarray(a_sol.get("net", [0] * T), dtype=float).ravel()
                total_import_gwh += np.maximum(net, 0).sum() * dt_h / 1e9

    print("wastage_gwh =", total_wastage_gwh)
    print("import_gwh =", total_import_gwh)